# PTB-XL 543-feature extraction pipeline

Companion code for the manuscript *Interpretable by construction: a CPU-only gradient-boosting pipeline with 543 handcrafted 12-lead ECG features for single- and multi-label classification on PTB-XL*.

This notebook is the exact feature-extraction code used for the paper. It reads the PTB-XL v1.0.3 500 Hz WFDB records, cleans each lead with NeuroKit2 `ecg_clean` (0.5 Hz high-pass Butterworth + powerline filter), extracts the per-record feature table in six families (morphology, intervals and QTc, complexity statistics, fragmented QRS, global indices, HRV), applies the model-ready quality control and column removal, merges the fQRS add-on, and attaches `patient_id` and the official `strat_fold`.

**Pipeline (five steps, Runtime → Run all):**

| Step | Output | Size in the paper |
|---|---|---|
| 1 | `ECG_Master_Features_Fixed_File_FINAL.csv` | 20,970 records × 555 columns (≈72 min, 2-core CPU) |
| 2 | `ECG_Master_Features_MODEL_ready.csv` | 20,310 records (660 quality failures removed) |
| 3 | `ECG_fQRS_Features.csv` | fragmented-QRS add-on (≈48 min) |
| 4 | merge | |
| 5 | **`ECG_MODEL_ready_FINAL_with_StratFold.csv`** | input of the modelling notebook |

The seven label-derived columns (`Disease_Name`, `Disease_Label_Encoded`, `Disease_NORM/MI/STTC/CD/HYP`) are present in the output for bookkeeping and **must be dropped before training** (Section 3.4 of the paper). After dropping them, 543 features remain.

**Inputs you must provide** (set the paths in the CONFIG cell):
- `records500.rar` (or an already extracted `records500/` folder) from PTB-XL v1.0.3, https://physionet.org/content/ptb-xl/1.0.3/
- `ptbxl_database.csv` and `scp_statements.csv` from the same release
- the primary-label metadata file `ptbxl_primary_NORM_MI_STTC_HYP_CD_labeled_metadata.csv` (built from `ptbxl_database.csv` with the superclass map, see Supplementary Note S1)

Pinned environment: Python 3.10+, `numpy==2.0.2`, `pandas==2.2.3`, `neurokit2==0.2.13`, `wfdb`, `scipy`, `tqdm`, `patool`.

Inline comments are partly in Bengali (the authors' working language); the code paths, constants and function names are self-explanatory and unchanged from the run that produced the paper's numbers.


## 1. Environment (Colab)

In [ ]:
# Mount Google Drive when running in Colab (skip silently elsewhere)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

In [ ]:
!pip -q install "numpy==2.0.2" "pandas==2.2.3" "neurokit2==0.2.13" wfdb mne patool tqdm scipy pyarrow

## 2. CONFIG

Edit the four paths below to your own locations. Everything else is left exactly as in the paper run.
The main cell (Section 3) defines the same constants internally with the authors' Drive paths; the values set here override them.

In [ ]:
import os
BASE_DIR      = "/content/drive/MyDrive/PTB-XL Dataset Analysis"   # folder with the inputs, outputs are written here
RAR_FILE_PATH = os.path.join(BASE_DIR, "records500.rar")             # PTB-XL 500 Hz WFDB archive
METADATA_CSV  = os.path.join(BASE_DIR, "ptbxl_primary_NORM_MI_STTC_HYP_CD_labeled_metadata.csv")
PTBXL_DB_CSV  = os.path.join(BASE_DIR, "ptbxl_database.csv")
os.makedirs(BASE_DIR, exist_ok=True)
print("BASE_DIR:", BASE_DIR)

## 3. Full pipeline (verbatim code of the paper run)

Steps 1 to 5 run in sequence with checkpointing every 500 records, so an interrupted run resumes.

In [ ]:
# -*- coding: utf-8 -*-
"""
PTB-XL ONE-STEP FINAL FEATURE FILE BUILDER
===========================================
Consolidates the author's final fixed-file extractor (original notebook Cell 41),
model-ready QC/drop stage (Cell 49), fQRS add-on (Cell 52), and strat_fold merge.

Expected final output:
    ECG_MODEL_ready_FINAL_with_StratFold.csv

Run in Colab after installing the pinned dependencies, or choose Runtime > Run all
in the companion notebook.
"""
import os
import json
import shutil
from pathlib import Path

# Mount Google Drive automatically in Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

"""
PTB-XL 12-Lead ECG Feature Extraction Pipeline
==============================================

উদ্দেশ্য: PTB-XL-এর 12-লিড ECG রেকর্ড (WFDB, 500 Hz, 10 সেকেন্ড) থেকে
রেকর্ডপ্রতি ৫০০+ ফিচার নিষ্কাশন — NORM/MI/STTC/CD/HYP পাঁচ-শ্রেণির
রোগ-শ্রেণিবিন্যাস গবেষণার জন্য। ইনপুট পড়া হয় আসল WFDB ফরম্যাট থেকে সরাসরি
(wfdb.rdrecord, শারীরিক একক mV) — কোনো মধ্যবর্তী রূপান্তর-ফরম্যাট ব্যবহৃত হয় না।

সংকেত-স্থাপত্য — দুটি স্বতন্ত্র রেফারেন্স-পরিবার, কখনো মিশ্রিত হয় না:
  • clean-পাথ (NeuroKit ecg_clean): R-peak শনাক্তকরণ, wave delineation, সব
    টাইমিং-মেট্রিক (PR/QRS/QT/ST/T intervals) ও সংকেত-পরিসংখ্যান।
  • morph-পাথ (median-filter baseline-wander subtraction, van Alsté-ধাঁচের
    0.25s ও 0.75s কার্নেল): সব প্রাবল্য-মেট্রিক (R/Q/S/P/T amplitude, QRS net
    area, fQRS প্রক্সি, V1 P-terminal force) — এই পথে QRS-শিখর ক্ষয়হীন থাকে।

মূল পদ্ধতি:
  • Delineation: prominence-পদ্ধতি (max_qrs_interval=260ms, max_pr_interval=400ms)।
  • ST/T-মাপ: প্রতি লিডে median-beat টেমপ্লেটের ওপর — flattest-উইন্ডো baseline,
    slope-collapse J-বিন্দু (armed-গেটসহ), T-onset-সীমায় clamped ST-point,
    টেমপ্লেট T-peak amplitude। টেমপ্লেট সম্ভব না হলে (<4 beat) honest NaN।
  • প্রকৃত Q/S-তরঙ্গের genuineness মাস্ক: R-সীমানা থেকে ≤60ms এবং isoelectric
    baseline-সম্পর্কিত গভীরতা ≥0.10 mV — delineator-এর ডিফল্ট শিখর-প্লেসমেন্টের
    পরিবর্তে প্রকৃত তরঙ্গ যাচাই।
  • ওয়াইড-QRS সংরক্ষণ: প্রচলিত QRS duration-এর পাশাপাশি প্রতি বীটে
    ডেরিভেটিভ-ভিত্তিক প্রসারিত অফসেট (নয়েজ-অভিযোজিত থ্রেশহোল্ড, mid-QRS
    resume-চেক, T-onset গার্ড); চূড়ান্ত মান দুটোর বড়টি — BBB-তেও সম্পূর্ণ
    প্রস্থ ধরা যায়।
  • Frontal QRS axis: leads I ও aVF-এর net QRS area থেকে atan2।
  • V1 PTFV1 (Morris): P-উইন্ডোর টার্মিনাল-নেগেটিভ অংশের গভীরতা×স্থিতিকাল।

সীমাবদ্ধতা (পেপারে উল্লেখ্য):
  ১) QRS onset-পার্শ্বের initial forces (যেমন LBBB-এর প্রাথমিক ধীর অংশ)
     রক্ষণশীলভাবেই মাপা হয় — প্রস্থ সামান্য কম আসতে পারে।
  ২) P-তরঙ্গের onset/offset নির্ভুলতা মাঝারি — PR_Interval ও PTFV1 lead-median
     দিয়ে প্রশমিত; PTFV1-এ N_Valid_Beats ≥ 3 ফিল্টারের পরামর্শ।
  ৩) bradycardia (<65 bpm) রেকর্ডে ST-মাপে ±0.15 mV-পর্যন্ত systematic bias
     সম্ভাব্য।
  ৪) Bazett-QTc উচ্চ হার্ট-রেটে ফুলায় — ব্যাখ্যায় Fridericia অগ্রাধিকার।

চালনা: এই ফাইল সরাসরি রান করলে সম্পূর্ণ ডেটাসেট এক্সট্রাকশন হয় (প্রতি ৫০০
রেকর্ডে checkpoint সেভ; ব্যাহত হলে পুনরায় চালু করলে স্বয়ংক্রিয় রিজিউম) এবং
শেষে চূড়ান্ত CSV তৈরি হয়।
"""

import os
import shutil
import warnings
import concurrent.futures
import numpy as np
import pandas as pd
import wfdb
import neurokit2 as nk
import patoolib
from scipy.stats import skew, kurtosis, entropy
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

warnings.filterwarnings("ignore")

LEAD_NAMES = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
REF_LEAD_FALLBACK_ORDER = ['II', 'I', 'V5', 'V2']

DELINEATE_METHOD = "prominence"
DELINEATE_KWARGS = {"max_qrs_interval": 260, "max_pr_interval": 400}  # ms

# --- Amplitude scale (WFDB p_signal সরাসরি mV-এককে) ---
AMPLITUDE_SCALE_CORRECTION = 1.0  # WFDB p_signal সরাসরি mV এককে
PLAUSIBLE_AMPLITUDE_MV_RANGE = (0.02, 50.0)

# --- v11: measurement-validity thresholds (clinical literature-grounded) ---
MAX_QS_EXTENSION_MS = 60              # genuine Q/S তরঙ্গ R-সীমানা থেকে সর্বোচ্চ এত ms দূরে
MIN_Q_DEPTH_MV = 0.10                 # measurable Q: কমপক্ষে 100 µV গভীর নিমজ্জন
MIN_S_DEPTH_MV = 0.10                 # একই যুক্তিতে প্রকৃত S-এর depth শর্ত
MIN_R_FOR_RATIO_MV = 0.50             # Q/R ratio-র denominator floor (V1/aVR tiny-R blow-up প্রতিরোধ)
RATIO_DENOM_FLOOR_MV = 0.20           # global Ratio_* ফিচারের denominator floor
# V1 P-terminal (Morris) পরিমাপ-থ্রেশহোল্ড (1 mm = 0.1 mV):
PTFV1_POS_MIN_MV = 0.02      # প্রাথমিক positive ফেজের ন্যূনতম শিখর
PTFV1_NEG_MIN_MV = 0.02      # টার্মিনাল-নেগেটিভ ফেজের ন্যূনতম গভীরতা
PTFV1_NEG_MIN_DUR_S = 0.02   # biphasic-ফ্ল্যাগের ন্যূনতম নেগেটিভ-স্থিতিকাল (≥0.04s হলে ক্লাসিক্যাল আকর্ষণীয়)
T_INVERSION_DEPTH_MV = -0.10          # ক্লিনিক্যাল T inversion: ≥0.1 mV গভীরতা
ST_DEPRESSION_THRESHOLD_MV = -0.10   # ক্লিনিক্যাল ST depression: ≥0.1 mV
COUNT_EXCLUDE_LEADS = {"aVR"}         # aVR-এ reciprocal ST/T পরিবর্তন নর্মাল — count থেকে বাদ

# পাথ কনফিগারেশন
RAR_FILE_PATH = "/content/drive/MyDrive/PTB-XL Dataset Analysis/records500.rar"
EXTRACT_DIR = "extracted_wfdb"
METADATA_CSV = "/content/drive/MyDrive/PTB-XL Dataset Analysis/ptbxl_primary_NORM_MI_STTC_HYP_CD_labeled_metadata.csv"
OUTPUT_CSV = "/content/drive/MyDrive/PTB-XL Dataset Analysis/ECG_Master_Features_Fixed_Final_File.csv"

SAVE_EVERY_N = 500
DISEASE_CLASSES = ["CD", "HYP", "MI", "NORM", "STTC"]

# --- Physiological bounds (standard 12-lead ECG reference range অনুযায়ী tightened) ---
BOUNDS = {
    "RR_Interval": (0.25, 2.5),
    "PR_Interval": (0.08, 0.40),
    "PR_Segment": (0.00, 0.30),
    "QRS_Duration": (0.04, 0.26),
    "P_Duration": (0.04, 0.20),
    "QT_Interval": (0.20, 0.60),
    "ST_Interval": (0.02, 0.30),
    "T_Wave_Duration": (0.05, 0.35),
    "RS_Ratio": (0.0, 30.0),
    "Q_R_Ratio": (0.0, 2.0),          # genuine-Q মাস্ক ও R-floor সক্রিয় থাকলে এর ওপরে মান আসে না
    "PR_Baseline": (-2.0, 2.0),       # চরম outlier-beat শনাক্তকরণ; সুষ্ঠু সরণ lead-median baseline-ই শোষণ করে
    "ST_Deviation": (-3.0, 3.0),
}
AMPLITUDE_BOUND_MV = (-5.0, 5.0)
# ============================================================
# পদ্ধতিগত নোট (বিস্তারিত মডিউল-ডকস্ট্রিং-এ):
#   • Frontal_QRS_Axis_Degrees = atan2(aVF-এর net QRS area, I-এর net QRS area);
#     area = per-beat integral-এর lead-median (beat-local baseline-বিয়োগকৃত)।
#   • V1 P-terminal force (Morris PTFV1) ও biphasic-P মার্কার — থ্রেশহোল্ড
#     নিচে কনস্ট্যান্টে; মান-পতাকা V1_PTFV1_N_Valid_Beats।
#   • ওয়াইড-QRS সংরক্ষণ: প্রচলিত duration-এর পাশাপাশি ডেরিভেটিভ-ভিত্তিক
#     প্রসারিত অফসেট; চূড়ান্ত মান দুটোর বড়টি।
#   • একই রেফারেন্স-পরিবারের নিয়ম: ভিন্ন সিগন্যালের zero কখনো মেশানো হয় না।
# ============================================================
PIPELINE_ID = "fixed_file"  # সংজ্ঞা-ছাপ — সহায়ক যাচাই-স্ক্রিপ্ট এই ছাপে সঠিক ফাইল যাচাই করে


# ============================================================
# হেল্পার ফাংশন সমূহ
# ============================================================

def _safe_median(values):
    arr = np.asarray(values, dtype=float)
    arr = arr[~np.isnan(arr)]
    return float(np.median(arr)) if len(arr) > 0 else np.nan


def _safe_median_min_n(values, min_n=3):
    """কমপক্ষে min_n=3 valid measurement না থাকলে NaN — একটামাত্র সম্ভাব্য-
    corrupted beat-কে বিশ্বাস না করে honest missingness — এটাই নীতি।"""
    arr = np.asarray(values, dtype=float)
    arr = arr[~np.isnan(arr)]
    return float(np.median(arr)) if len(arr) >= min_n else np.nan


def _apply_bounds(arr, low, high):
    arr = np.asarray(arr, dtype=float)
    return np.where((arr < low) | (arr > high), np.nan, arr)


def _match_lead_indices(ch_names):
    normalized = {name.strip().upper(): i for i, name in enumerate(ch_names)}
    return {lead: normalized[lead.upper()] for lead in LEAD_NAMES if lead.upper() in normalized}


def _safe_div(v1, v2, min_denominator=None):
    """min_denominator: ছোট/অস্থির denominator (যেমন V1/aVR-এ near-zero R)
    থেকে বিস্ফোরিত ratio এড়াতে floor; floor-এর নিচে হলে NaN।"""
    if pd.isna(v1) or pd.isna(v2) or v2 == 0:
        return np.nan
    if min_denominator is not None and abs(v2) < min_denominator:
        return np.nan
    return float(v1 / v2)


def safe_corr(x, y):
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    try:
        return float(np.corrcoef(x, y)[0, 1])
    except Exception:
        return np.nan


def _detect_reference_peaks(signals_mv, lead_to_idx, fs):
    for lead in REF_LEAD_FALLBACK_ORDER:
        if lead not in lead_to_idx:
            continue
        try:
            raw_signal = signals_mv[lead_to_idx[lead]]
            cleaned = nk.ecg_clean(raw_signal, sampling_rate=fs)
            _, info = nk.ecg_peaks(cleaned, sampling_rate=fs, correct_artifacts=True)
            r_peaks = info["ECG_R_Peaks"]
            if len(r_peaks) >= 5:
                return lead, cleaned, r_peaks
        except Exception:
            continue
    return None, None, None


def _per_beat_array(waves, key, n_beats):
    vals = np.asarray(waves.get(key, []), dtype=float)
    if len(vals) < n_beats:
        vals = np.concatenate([vals, np.full(n_beats - len(vals), np.nan)])
    elif len(vals) > n_beats:
        vals = vals[:n_beats]
    return vals


def _amplitude_at_indices(clean_signal, indices):
    n = len(clean_signal)
    amps = []
    for idx in indices:
        if idx is None or pd.isna(idx):
            amps.append(np.nan)
            continue
        idx_int = int(idx)
        amps.append(clean_signal[idx_int] if 0 <= idx_int < n else np.nan)
    return np.array(amps)


def calculate_shannon_entropy(signal, bins=64):
    try:
        counts, _ = np.histogram(signal, bins=bins, density=True)
        prob = counts[counts > 0]
        return float(entropy(prob, base=2))
    except Exception:
        return np.nan


def compute_hjorth_parameters(signal):
    try:
        activity = float(np.var(signal))
        if activity == 0 or np.isnan(activity):
            return np.nan, np.nan, np.nan

        diff_1 = np.diff(signal)
        var_diff_1 = np.var(diff_1)
        mobility = float(np.sqrt(var_diff_1 / activity))
        if mobility == 0 or np.isnan(mobility):
            return activity, mobility, np.nan

        diff_2 = np.diff(diff_1)
        var_diff_2 = np.var(diff_2)
        if var_diff_1 == 0:
            return activity, mobility, np.nan

        mobility_diff = float(np.sqrt(var_diff_2 / var_diff_1))
        complexity = float(mobility_diff / mobility)
        return activity, mobility, complexity
    except Exception:
        return np.nan, np.nan, np.nan


# ============================================================
# রেকর্ড-লেভেল HRV (ultra-short-term valid metrics only)
# ============================================================

def compute_record_level_hrv(r_peaks, fs):
    init_keys = ["RR_Interval", "SDNN", "CVRR", "RMSSD", "pNN50"]
    default_feats = {k: np.nan for k in init_keys}
    n_beats = len(r_peaks)
    if n_beats < 5:
        return default_feats

    rr_raw = np.diff(r_peaks) / fs
    rr_bounded = _apply_bounds(rr_raw, *BOUNDS["RR_Interval"])
    valid_rr = rr_bounded[~np.isnan(rr_bounded)]

    if len(valid_rr) < 2:
        return default_feats

    rr_interval = _safe_median(rr_bounded)
    sdnn_val = float(np.std(valid_rr, ddof=1))
    mean_rr_val = np.mean(valid_rr)

    feats = {
        "RR_Interval": rr_interval,
        "SDNN": sdnn_val,
        "CVRR": float(sdnn_val / mean_rr_val) if mean_rr_val != 0 else np.nan,
    }

    diff_rr = np.diff(valid_rr)
    if len(diff_rr) > 0:
        feats["RMSSD"] = float(np.sqrt(np.mean(diff_rr**2)))
        feats["pNN50"] = float(np.sum(np.abs(diff_rr) > 0.05) / len(diff_rr) * 100)
    else:
        feats["RMSSD"], feats["pNN50"] = np.nan, np.nan

    return feats


# ============================================================
# লিড-ভিত্তিক ফিচার এক্সট্রাকশন (কোর ইঞ্জিন)
# ============================================================

def _local_r_amplitude(clean_signal_mv, r_peaks, fs, window_ms=25):
    """প্রতিটা লিডে reference সময়বিন্দুর চারপাশে (±25ms) local maximum খুঁজে
    সেই লিডের নিজস্ব R-শীর্ষ (v9-এ verified fix)।"""
    window = max(1, int(round(window_ms / 1000.0 * fs)))
    n = len(clean_signal_mv)
    amps = []
    for rp in r_peaks:
        lo = max(0, int(rp) - window)
        hi = min(n, int(rp) + window + 1)
        if hi > lo:
            local_idx = lo + int(np.argmax(clean_signal_mv[lo:hi]))
            amps.append(clean_signal_mv[local_idx])
    return np.array(amps, dtype=float)


from scipy.ndimage import median_filter as _ndi_median_filter


def _wander_removed_mv(signal_mv, fs):
    """Median-filter baseline-wander subtraction (van Alsté
    ধাঁচি): 0.25s kernel QRS/P সরায়, 0.75s kernel T-পর্যন্ত; sig−baseline
    ringing-মুক্ত isoelectric-রেফারেন্সড সিগন্যাল দেয় — 0.5Hz হাই-পাস
    (ecg_clean)-এর per-beat coherent ringing/মিস-জিরো সরণ (sandbox-এ যাচাইকৃত
    ~−0.25mV PR sag → +0.13–0.23mV মিথ্যা ST elevation) এড়ায়। শুধু ST/T
    টেমপ্লেট-পাথে ব্যবহৃত; বাকি সব ফিচার যাচাইকৃত ecg_clean-এই থাকে।"""
    sig = np.asarray(signal_mv, dtype=float)
    k1 = int(round(0.25 * fs)) | 1
    k2 = int(round(0.75 * fs)) | 1
    b = _ndi_median_filter(sig, size=k1, mode="nearest")
    b = _ndi_median_filter(b, size=k2, mode="nearest")
    return sig - b


def _build_beat_template(clean_signal_mv, r_peaks, fs):
    """R-peak-কেন্দ্রিক median-beat template (±3-sample correlation realignment সহ)।
    রিটার্ন: (tpl, r_idx, rr_med_samples) অথবা (None, None, None) যদি beat < 4."""
    r_peaks = np.asarray(r_peaks, dtype=float)
    r_peaks = r_peaks[~np.isnan(r_peaks)].astype(int)
    sig = np.asarray(clean_signal_mv, dtype=float)
    if len(r_peaks) < 4:
        return None, None, None
    rr_med = float(np.median(np.diff(r_peaks))) if len(r_peaks) > 1 else fs
    pre = int(round(0.20 * fs))
    post = int(round(min(0.45 * fs, 0.55 * rr_med)))
    if post <= int(0.25 * fs):          # খুব tachycardic — T দেখার জায়গাই নেই
        post = int(0.25 * fs)
    beats = []
    for rp in r_peaks:
        if rp - pre >= 0 and rp + post < len(sig):
            beats.append(sig[rp - pre: rp + post].astype(float))
    if len(beats) < 4:
        return None, None, None
    beats = np.array(beats)
    tpl = np.median(beats, axis=0)
    # ±3-sample realignment: R-জোনের ([−50ms, +80ms]) correlation সর্বোচ্চ করে
    lo, hi = pre - int(0.050 * fs), pre + int(0.080 * fs)
    ref = tpl[max(0, lo):hi]
    win = int(round(0.003 * fs))
    fixed = []
    for b in beats:
        best_s, best_c = 0, -np.inf
        for s in range(-win, win + 1):
            seg = b[max(0, lo + s): hi + s]
            if len(seg) != len(ref):
                continue
            c = float(np.dot(seg, ref))
            if c > best_c:
                best_c, best_s = c, s
        fixed.append(np.roll(b, -best_s) if best_s != 0 else b)
    tpl = np.median(np.array(fixed), axis=0)
    return tpl, pre, rr_med


def _flattest_baseline_tpl(tpl, r_idx, fs):
    """R-এর 20–80ms আগের জোনে সবচেয়ে ফ্ল্যাট 15ms উইন্ডোর median =
    isoelectric baseline (PR/TP segment)। P-tail থাকলেও flatness-গেট বেছে নেয়।"""
    w = int(round(0.015 * fs))
    lo = max(0, r_idx - int(round(0.080 * fs)))
    hi = max(w, r_idx - int(round(0.020 * fs)))
    best_i, best_score = None, np.inf
    for i in range(lo, max(lo + 1, hi - w)):
        seg = tpl[i:i + w]
        if len(seg) < w:
            continue
        score = np.max(np.abs(np.diff(seg))) + np.ptp(seg)
        if score < best_score:
            best_score, best_i = score, i
    if best_i is None:
        return np.nan
    return float(np.median(tpl[best_i:best_i + w]))


def _measure_template_st_t(tpl, r_idx, rr_med, fs, hr_bpm):
    """টেমপ্লেটের ওপর শারীরস্থানিক ST/T মাপ। রিটার্ন dict:
    baseline, qrs_on_tpl, j_tpl, t_onset_tpl, t_peak_tpl (সব abs index),
    ST_J, ST60, ST80 (mV, baseline-সম্পর্কিত, T-clamped), T_Amp (signed), ST_Slope।
    ব্যর্থ হলে None।"""
    n = len(tpl)
    if tpl is None or r_idx is None:
        return None
    baseline = _flattest_baseline_tpl(tpl, r_idx, fs)
    if np.isnan(baseline):
        return None

    amp = float(np.ptp(tpl[max(0, r_idx - int(0.08 * fs)): r_idx + int(0.08 * fs)]))
    amp = max(amp, 0.05)

    # ---- QRS offset (J): SLOPE-COLLAPSE সনাক্তকরণ — amplitude-নিরপেক্ষ, তাই
    # উত্তোলিত/অবনত ST shelf (+0.20/−0.15mV)-এও কাজ করে (GT-যাচাইকৃত)। আগের
    # amplitude-threshold সংস্করণ S-upstroke-র −0.12mV অব্ধি ভেতরেই fire করত।
    # পদ্ধতি: QRS-জোনের শেষ extremum (S-nadir থাকলে সেটা, না হলে R) এর পরে
    # |slope| QRS-max-এর ক্ষুদ্র ভগ্নাংশে টেকসইভাবে নেমে আসা প্রথম বিন্দুই J —
    # অর্থাৎ QRS সরে গিয়ে ST/TP plateau শুরু হওয়ার জায়গা।
    d = np.gradient(tpl)
    k = max(1, int(round(0.008 * fs)))          # 8ms boxcar স্মুদ
    d_sm = np.convolve(d, np.ones(k) / k, mode="same")
    q_lo = r_idx + int(round(0.010 * fs))
    q_hi = min(n - 3, r_idx + int(round(0.090 * fs)))
    segmin = int(np.argmin(tpl[q_lo:q_hi])) + q_lo
    if tpl[segmin] < baseline - 0.030:          # সত্যিকার S আছে → শেষ extremum = S-nadir
        anchor = segmin
    else:                                        # S নেই (qR/R প্যাটার্ন) → R-peak থেকেই
        anchor = r_idx
    ref_lo = max(0, r_idx - int(round(0.020 * fs)))
    max_s = float(np.percentile(np.abs(d_sm[ref_lo:q_hi]), 95))
    thr_d = max(0.04 * max_s, 0.010)            # mV/sample — QRS slope-এর 4%
    j = None
    run = 0
    start = anchor + int(round(0.002 * fs))
    end = min(n - 3, r_idx + int(round(0.130 * fs)))
    sust = max(2, int(round(0.006 * fs)))
    armed = False    # সত্যিকার QRS-descent (|d| ≥ thr) পূর্ব-শর্ত (armed-গেট)
                     # অন্তত একবার না দেখলে "কোলাপ্স" গোনা যাবে না — S-বিহীন
                     # লিডে anchor=r_idx হলে apex-এর ফ্ল্যাট ধরাটাই J হয়ে যেত
    for i in range(start, end):
        if not armed:
            if abs(d_sm[i]) >= thr_d:
                armed = True
            continue
        if abs(d_sm[i]) < thr_d:
            run += 1
            if run >= sust:
                j = i - run + 1
                break
        else:
            run = 0
    if j is None:
        j = int(r_idx + min(int(round(0.070 * fs)), int(round(0.30 * rr_med))))
    j = int(j)

    # ---- T-peak: [J+40ms, min(template শেষ, R+0.45s, R+0.55*RR)] জোনে
    t_lo = j + int(round(0.040 * fs))
    t_hi = min(n - 2, r_idx + int(round(min(0.45 * fs, 0.55 * rr_med))))
    if t_hi <= t_lo + int(round(0.03 * fs)):
        return None                              # T দেখার জায়গা নেই → honest bail
    seg = tpl[t_lo:t_hi] - baseline
    t_pk = t_lo + int(np.argmax(np.abs(seg)))
    t_amp = float(tpl[t_pk] - baseline)

    # ---- T-onset: T-peak থেকে পেছনে — |tpl−baseline| ছোট হওয়া শেষ বিন্দু
    thr_t = max(0.025, 0.08 * abs(t_amp))
    t_on = t_lo
    for i in range(t_pk, max(j + 2, t_lo), -1):
        if abs(tpl[i] - baseline) <= thr_t:
            t_on = i
            break
    t_on = max(t_on, j + int(round(0.010 * fs)))

    # ---- ST পয়েন্ট: T-onset−16ms ক্রস করা নিষেধ; হাই-রেটে J+60 প্রাথমিক
    margin = int(round(0.016 * fs))
    floor = j + int(round(0.010 * fs))
    cap = max(t_on - margin, floor)
    off60 = int(round(0.060 * fs)) if hr_bpm is None or hr_bpm >= 100 else int(round(0.060 * fs))
    st60_pt = int(min(j + off60, cap))
    st80_pt = int(min(j + int(round(0.080 * fs)), cap))
    if st60_pt <= j:
        st60_pt = int(min(cap, j + max(int(round(0.006 * fs)), (t_on - j) // 2)))
    st80_pt = max(st80_pt, st60_pt)

    st_j = float(tpl[j] - baseline)
    st60 = float(tpl[st60_pt] - baseline)
    st80 = float(tpl[st80_pt] - baseline)
    dt = max((st80_pt - j) / fs, 1e-3)
    st_slope = float((tpl[st80_pt] - tpl[j]) / dt)

    return {
        "baseline": baseline, "j_idx": j, "t_on_idx": t_on, "t_pk_idx": t_pk,
        "ST_J": st_j, "ST60": st60, "ST80": st80, "T_Amp": t_amp,
        "ST_Slope": st_slope,
        "j_ms": (j - r_idx) / fs * 1000.0,
        "st80_pt_ms": (st80_pt - r_idx) / fs * 1000.0,
    }


def _p_terminal_measure_v1(p_seg, fs, pos_min=PTFV1_POS_MIN_MV, neg_min=PTFV1_NEG_MIN_MV,
                           neg_dur_min=PTFV1_NEG_MIN_DUR_S):
    """একটি বীটের V1 P-window (P_Onset→P_Offset) এর Morris P-terminal পরিমাপ।

    নিয়ম (morph-পরিবারের zero-রেফারেন্সে, amp_sig-এর স্লাইস):
      - প্রাথমিক positive শিখর < pos_min হলে → পরিমাপযোগ্য নয় (সব 0)।
      - শিখরের পরে global minimum ≤ −neg_min হলে, সেই মিনিমামের চারপাশের
        contiguous sub-zero রান = টার্মিনাল-নেগেটিভ অংশ:
        PTFV1 = গভীরতা(mV)×10(mm)×স্থিতিকাল(s) → ঋণাত্মক মান (Morris-কনভেনশন)।
      - biphasic-ফ্ল্যাগের অতিরিক্ত শর্ত: রান-স্থিতিকাল ≥ neg_dur_min।
      - monophasic-negative/flat P: Morris-সংজ্ঞা প্রযোজ্য নয় → 0 (পরিমাপযোগ্য নয়)।
    """
    res = {"ptfv1_mm_s": 0.0, "term_neg_depth_mv": 0.0, "term_neg_dur_s": 0.0, "biphasic": 0}
    p_seg = np.asarray(p_seg, dtype=float)
    if p_seg.size < 3:
        return res
    imax = int(np.argmax(p_seg))
    if p_seg[imax] < pos_min:
        return res
    imin = imax + int(np.argmin(p_seg[imax:]))
    if p_seg[imin] > -neg_min:
        return res
    zs = imin
    while zs - 1 > imax and p_seg[zs - 1] < 0:
        zs -= 1
    ze = imin
    while ze + 1 < p_seg.size and p_seg[ze + 1] < 0:
        ze += 1
    dur_s = (ze - zs + 1) / float(fs)
    depth_mv = float(p_seg[imin])
    res["ptfv1_mm_s"] = depth_mv * 10.0 * dur_s
    res["term_neg_depth_mv"] = depth_mv
    res["term_neg_dur_s"] = dur_s
    res["biphasic"] = int(depth_mv <= -neg_min and dur_s >= neg_dur_min)
    return res


def _extended_qrs_offset(sig_mv, r_idx, end_idx, fs, max_slope_frac=0.05,
                         slope_floor_mv=0.002, sustain_ms=12.0, box_ms=10.0,
                         resume_frac=0.08):
    """R-শীর্ষের পরে QRS-কার্যকলাপ কোথায় থামে — ডেরিভেটিভ-ভিত্তিক অনুসন্ধান।

    নিয়ম: |d(sig)|-এর 10ms বক্স-স্মুদ মান থ্রেশহোল্ডের নিচে ≥12ms টিকলে অফসেট।
    থ্রেশহোল্ড = max(5%×উইন্ডো-চরম স্লোপ, নয়েজ-অভিযোজিত ফ্লোর) — ফ্লোর একই বীটের
    isoelectric প্রি-R উইন্ডোর (R−50..R−14ms) median+4×MAD থেকে, যাতে নয়েজি
    বীটেও অকাল বা দেরিতে থামা না ঘটে। RSR′-ধাঁচের mid-QRS নীরব ফাঁক থেকে বাঁচতে:
    নিম্ন-কার্যকলাপ-রানের পরে স্লোপ আবার resume_frac×চরম ছাড়ালে সেটি চূড়ান্ত
    নয় বলে অতিক্রম করা হয়। থামা না পেলে উইন্ডো-প্রান্ত (উর্ধ্বসীমা সুরক্ষা)।
    """
    r_idx = int(r_idx)
    end_idx = int(min(end_idx, len(sig_mv) - 1))
    if end_idx - r_idx < int(round(0.040 * fs)):
        return None
    win = np.asarray(sig_mv[r_idx:end_idx], dtype=float)
    k = max(1, int(round(box_ms / 1000.0 * fs)))
    _box = np.full(k, 1.0 / k)
    ds = np.convolve(np.abs(np.diff(win)), _box, mode="same")
    mx = float(ds.max())
    if not np.isfinite(mx) or mx <= 0:
        return None
    # বীট-অভিযোজিত নয়েজ-ফ্লোর: প্রি-R isoelectric উইন্ডো (R−60..R−10ms)
    p_hi = max(0, r_idx - int(round(0.014 * fs)))
    p_lo = max(0, r_idx - int(round(0.050 * fs)))
    floor = slope_floor_mv
    if p_hi - p_lo >= 10:
        nds = np.convolve(np.abs(np.diff(np.asarray(sig_mv[p_lo:p_hi], dtype=float))), _box, mode="same")
        if nds.size:
            n_med = float(np.median(nds))
            n_mad = float(np.median(np.abs(nds - n_med)))
            floor = max(slope_floor_mv, n_med + 4.0 * n_mad)
    thr = max(max_slope_frac * mx, floor)
    resume_thr = max(1.5 * thr, resume_frac * mx)
    sustain = max(2, int(round(sustain_ms / 1000.0 * fs)))
    below = ds < thr
    i = 0
    n = len(below)
    while i < n:
        if not below[i]:
            i += 1
            continue
        j = i
        while i < n and below[i]:
            i += 1
        run_len = i - j
        if run_len >= sustain:
            # mid-QRS ফাঁক কিনা যাচাই: পরে জোরালো কার্যকলাপ ফিরলে এড়িয়ে যাই
            if i < n and float(ds[i:].max()) > resume_thr:
                continue
            return r_idx + j
    return r_idx + len(win) - 1


def extract_lead_features(clean_signal_mv, r_peaks, fs, lead_name, hrv_rr_interval, morph_signal_mv=None):
    feats = {}
    prefix = lead_name
    n_beats = len(r_peaks)
    # অ্যাম্প্লিচুড-পরিবারের রেফারেন্স সিগন্যাল (শিখর-ক্ষয়হীন morph)।
    # না পেলে প্রাপ্ত clean_signal থেকেই wander-sub (fallback)।
    amp_sig = morph_signal_mv if morph_signal_mv is not None else _wander_removed_mv(clean_signal_mv, fs)

    feats[f"{prefix}_Shannon_Entropy"] = calculate_shannon_entropy(clean_signal_mv, bins=64)
    feats[f"{prefix}_Amplitude_Entropy"] = calculate_shannon_entropy(np.abs(clean_signal_mv), bins=64)
    feats[f"{prefix}_ZCR"] = int(((clean_signal_mv[:-1] * clean_signal_mv[1:]) < 0).sum())
    feats[f"{prefix}_RMS"] = float(np.sqrt(np.mean(clean_signal_mv**2)))
    feats[f"{prefix}_Range"] = float(np.ptp(clean_signal_mv))
    feats[f"{prefix}_Skewness"] = float(skew(clean_signal_mv))
    feats[f"{prefix}_Kurtosis"] = float(kurtosis(clean_signal_mv))
    feats[f"{prefix}_Signal_Energy"] = float(np.sum(clean_signal_mv**2))

    activity, mobility, complexity = compute_hjorth_parameters(clean_signal_mv)
    feats[f"{prefix}_Hjorth_Activity"] = activity
    feats[f"{prefix}_Hjorth_Mobility"] = mobility
    feats[f"{prefix}_Hjorth_Complexity"] = complexity

    init_cols = ["N_Valid_Beats", "QT_Interval", "PR_Interval", "PR_Segment", "QRS_Duration", "P_Duration",
                 "ST_Interval", "T_Wave_Duration", "T_Wave_Slope", "P_Amplitude", "Q_Amplitude",
                 "R_Amplitude", "S_Amplitude", "T_Amplitude", "ST_Level", "ST80_Level", "ST_J_Level", "ST_Slope",
                 "RS_Ratio", "Fragmented_QRS_Fraction", "T_Amplitude_Variability",
                 "Q_R_Ratio", "Delineate_Error", "QRS_Net_Area", "QRS_Onset_Rel_R", "QRS_Offset_Rel_R",
                 "QRS_Duration_Extended", "QRS_Offset_Rel_R_Extended"]
    # T_Peak_Amp (ডুপ্লিকেট) ও T_Offset_Amp (near-constant) বাদ
    for col in init_cols:
        feats[f"{prefix}_{col}"] = np.nan

    feats[f"{prefix}_N_Valid_Beats"] = 0

    if lead_name == "V1":  # delineation ব্যর্থ হলেও কলাম উপস্থিত থাকে
        feats["V1_PTFV1_N_Valid_Beats"] = 0
        for _vc in ["V1_P_Terminal_Force_mm_s", "V1_P_Terminal_Neg_Depth_mV",
                    "V1_P_Terminal_Neg_Duration_s", "V1_P_Biphasic_Fraction", "V1_P_Biphasic_Flag"]:
            feats[_vc] = np.nan

    if n_beats > 0:
        r_amps_bounded = _apply_bounds(_local_r_amplitude(amp_sig, r_peaks, fs), *AMPLITUDE_BOUND_MV)
        feats[f"{prefix}_R_Amplitude"] = _safe_median(r_amps_bounded)

    try:
        _, waves = nk.ecg_delineate(
            clean_signal_mv, r_peaks, sampling_rate=fs,
            method=DELINEATE_METHOD, **DELINEATE_KWARGS
        )

        p_onsets = _per_beat_array(waves, "ECG_P_Onsets", n_beats)
        p_offsets = _per_beat_array(waves, "ECG_P_Offsets", n_beats)
        r_onsets = _per_beat_array(waves, "ECG_R_Onsets", n_beats)
        r_offsets = _per_beat_array(waves, "ECG_R_Offsets", n_beats)
        t_onsets = _per_beat_array(waves, "ECG_T_Onsets", n_beats)
        t_offsets = _per_beat_array(waves, "ECG_T_Offsets", n_beats)

        p_peaks = _per_beat_array(waves, "ECG_P_Peaks", n_beats)
        q_peaks = _per_beat_array(waves, "ECG_Q_Peaks", n_beats)
        s_peaks = _per_beat_array(waves, "ECG_S_Peaks", n_beats)
        t_peaks = _per_beat_array(waves, "ECG_T_Peaks", n_beats)

        max_ext = int(round(MAX_QS_EXTENSION_MS / 1000.0 * fs))  # 60ms

        def _extend_boundary(base, candidate, direction):
            """প্রকৃত QRS onset/offset: candidate (q_peak/s_peak) কেবল
            base (r_onset/r_offset) থেকে ≤60ms দূরত্বে থাকলে গৃহীত হয়;
            base NaN হলে boundary NaN-ই থাকে — 60ms cap কখনো এড়ানো হয় না।"""
            out = np.copy(base)
            for i in range(len(base)):
                b, c = base[i], candidate[i]
                if np.isnan(b) or np.isnan(c):
                    continue
                dist = (b - c) if direction == 1 else (c - b)
                if 0 <= dist <= max_ext:
                    out[i] = c
            return out

        with np.errstate(invalid='ignore'):
            qrs_onsets = _extend_boundary(r_onsets, q_peaks, direction=1)
            qrs_offsets = _extend_boundary(r_offsets, s_peaks, direction=-1)

        # ST/T টেমপ্লেট-যন্ত্রাণী (tpl_m) — morph (wander-subtracted) পরিবারে
        # সিগন্যালের ওপর। এটা শুধু ST_J/ST60/ST80/ST_Slope/T_Amplitude-এ ব্যবহৃত।
        _hr_bpm = (60.0 / hrv_rr_interval) if (not pd.isna(hrv_rr_interval) and hrv_rr_interval > 0) else None
        _tpl_src = amp_sig   # একই morph রেফারেন্স
        _tpl, _tpl_ridx, _tpl_rr = _build_beat_template(_tpl_src, r_peaks, fs)
        tpl_m = _measure_template_st_t(_tpl, _tpl_ridx, _tpl_rr, fs, _hr_bpm) if _tpl is not None else None

        # Q/S-এর isoelectric baseline: clean-সিগন্যালের per-beat PR-উইন্ডোম মেডিয়ানের
        # lead-median (scalar)। মূল নিয়ম — একই রেফারেন্স-পরিবারের মাপ একসাথে থাকে:
        # Q/S↔clean-ভিত্তিক baseline, ST/T↔টেমপ্লেট-অভ্যন্তরীণ পার্থক্য; দুই ভিন্ন
        # সিগন্যালের zero-রেফারেন্স কখনো মেশানো হয় না।
        pre_q_lo = int(round(0.040 * fs))
        pre_q_hi = int(round(0.010 * fs))
        baseline_beats = []
        for q_on in qrs_onsets:
            if np.isnan(q_on):
                baseline_beats.append(np.nan)
                continue
            hi = int(q_on) - pre_q_hi
            lo = max(0, int(q_on) - pre_q_lo)
            baseline_beats.append(float(np.median(clean_signal_mv[lo:hi])) if hi > lo else np.nan)
        baseline_beats = _apply_bounds(np.array(baseline_beats), *BOUNDS["PR_Baseline"])
        pr_baseline_lead = _safe_median_min_n(baseline_beats)
        if pd.isna(pr_baseline_lead):
            pr_baseline_lead = np.nan

        # genuine Q/S মাস্ক — prominence delineator প্রকৃত wave না থাকলেও
        # Q_Peak/S_Peak বসিয়ে দেয়, তাই দুটি শর্তে যাচাই: (ক) q_peak/s_peak
        # R-boundary-র ≤60ms-এর মধ্যে, (খ) নিমজ্জন isoelectric-সম্পর্কিত
        # (per-beat PR baseline-বিয়োগকৃত) কমপক্ষে 0.10 mV গভীর। গভীরতা
        # baseline-সম্পর্কিত — যেন স্থির baseline offset মিথ্যা depth না হয়।
        with np.errstate(invalid='ignore'):
            q_dist = r_onsets - q_peaks      # onset-দিক: q, r-এর আগে → ধনাত্মক দূরত্ব
            s_dist = s_peaks - r_offsets     # offset-দিক: s, r-এর পরে → ধনাত্মক দূরত্ব
            genuine_q_pos = (q_dist >= 0) & (q_dist <= max_ext)
            genuine_s_pos = (s_dist >= 0) & (s_dist <= max_ext)

        q_raw = _amplitude_at_indices(amp_sig, q_peaks)
        s_raw = _amplitude_at_indices(amp_sig, s_peaks)

        with np.errstate(invalid='ignore'):
            q_rel = q_raw - pr_baseline_lead
            s_rel = s_raw - pr_baseline_lead

        genuine_q = genuine_q_pos & (q_rel <= -MIN_Q_DEPTH_MV)
        genuine_s = genuine_s_pos & (s_rel <= -MIN_S_DEPTH_MV)

        # প্রতিটা beat-এ তিনতর অবস্থা: প্রকৃত Q/S থাকলে baseline-সম্পর্কিত amplitude;
        # R-boundary ও lead-baseline দুটোই আছে কিন্তু প্রকৃত wave নেই → 0.0;
        # delineation/baseline অনুপস্থিত → NaN (অজানা)। baseline lead-median
        # scalar — একটি beat-এর অনুপস্থিতি অন্য beat-এর মান নষ্ট করে না।
        baseline_ok = not pd.isna(pr_baseline_lead)
        with np.errstate(invalid='ignore'):
            q_amp_beats = np.where(
                (~np.isnan(r_onsets)) & baseline_ok,
                np.where(genuine_q, q_rel, 0.0),
                np.nan,
            )
            s_amp_beats = np.where(
                (~np.isnan(r_offsets)) & baseline_ok,
                np.where(genuine_s, s_rel, 0.0),
                np.nan,
            )

        valid_mask = ~np.isnan(qrs_onsets) & ~np.isnan(qrs_offsets)
        feats[f"{prefix}_N_Valid_Beats"] = int(np.sum(valid_mask))

        feats[f"{prefix}_QT_Interval"] = _safe_median_min_n(_apply_bounds((t_offsets - qrs_onsets) / fs, *BOUNDS["QT_Interval"]))
        feats[f"{prefix}_PR_Interval"] = _safe_median_min_n(_apply_bounds((qrs_onsets - p_onsets) / fs, *BOUNDS["PR_Interval"]))
        feats[f"{prefix}_PR_Segment"] = _safe_median_min_n(_apply_bounds((qrs_onsets - p_offsets) / fs, *BOUNDS["PR_Segment"]))
        feats[f"{prefix}_QRS_Duration"] = _safe_median_min_n(_apply_bounds((qrs_offsets - qrs_onsets) / fs, *BOUNDS["QRS_Duration"]))
        feats[f"{prefix}_P_Duration"] = _safe_median_min_n(_apply_bounds((p_offsets - p_onsets) / fs, *BOUNDS["P_Duration"]))
        feats[f"{prefix}_ST_Interval"] = _safe_median_min_n(_apply_bounds((t_onsets - qrs_offsets) / fs, *BOUNDS["ST_Interval"]))
        feats[f"{prefix}_T_Wave_Duration"] = _safe_median_min_n(_apply_bounds((t_offsets - t_onsets) / fs, *BOUNDS["T_Wave_Duration"]))

        # ওয়াইড-QRS প্রসারিত অফসেট (ডেরিভেটিভ-ভিত্তিক) — প্রচলিত মেট্রিক অক্ষত;
        # T-onset-এর 4ms আগে পর্যন্ত সর্বোচ্চ, ঊর্ধ্বসীমা R+200ms।
        ext_offsets = np.full(n_beats, np.nan)
        _max_ext_search = int(round(0.200 * fs))
        _t_guard = max(1, int(round(0.004 * fs)))
        for _bi in range(n_beats):
            if np.isnan(qrs_onsets[_bi]):
                continue
            _end = int(r_peaks[_bi]) + _max_ext_search
            if not np.isnan(t_onsets[_bi]):
                _end = min(_end, int(t_onsets[_bi]) - _t_guard)
            _off = _extended_qrs_offset(amp_sig, int(r_peaks[_bi]), _end, fs)
            if _off is not None:
                ext_offsets[_bi] = _off
        # চূড়ান্ত মান: per-beat max(প্রচলিত, প্রসারিত); NaN-সহায়ক নিয়ম
        qrs_offsets_hybrid = np.where(
            np.isnan(qrs_offsets), ext_offsets,
            np.where(np.isnan(ext_offsets), qrs_offsets, np.maximum(qrs_offsets, ext_offsets)))
        feats[f"{prefix}_QRS_Duration_Extended"] = _safe_median_min_n(
            _apply_bounds((qrs_offsets_hybrid - qrs_onsets) / fs, *BOUNDS["QRS_Duration"]))
        _off_rel_ext = [(hi - rp) / fs for hi, rp in zip(qrs_offsets_hybrid, r_peaks)
                        if not np.isnan(hi)]
        feats[f"{prefix}_QRS_Offset_Rel_R_Extended"] = _safe_median_min_n(_off_rel_ext)

        qrs_areas = []
        for lo, hi in zip(qrs_onsets, qrs_offsets):
            if not np.isnan(lo) and not np.isnan(hi) and hi > lo:
                # beat-local baseline বিয়োগ — একই morph-সিগন্যালের PR-উইন্ডো থেকে
                _blo = int(lo)
                _bhi = _blo - pre_q_hi
                _blw = max(0, _blo - pre_q_lo)
                _bb = float(np.median(amp_sig[_blw:_bhi])) if _bhi > _blw else 0.0
                qrs_areas.append(float(np.sum(amp_sig[_blo:int(hi)] - _bb) / fs))
        feats[f"{prefix}_QRS_Net_Area"] = _safe_median_min_n(qrs_areas)

        onset_rel_r = [(lo - rp) / fs for lo, rp in zip(qrs_onsets, r_peaks) if not np.isnan(lo)]
        offset_rel_r = [(hi - rp) / fs for hi, rp in zip(qrs_offsets, r_peaks) if not np.isnan(hi)]
        feats[f"{prefix}_QRS_Onset_Rel_R"] = _safe_median_min_n(onset_rel_r)
        feats[f"{prefix}_QRS_Offset_Rel_R"] = _safe_median_min_n(offset_rel_r)

        # (pr_baseline_lead উপরে গণনা করা হয়েছে)

        def _amp_feature(peak_indices):
            amps = _amplitude_at_indices(amp_sig, peak_indices)
            return _safe_median_min_n(_apply_bounds(amps, *AMPLITUDE_BOUND_MV))

        feats[f"{prefix}_P_Amplitude"] = _amp_feature(p_peaks)

        # V1 P-terminal force (Morris) + biphasic-P — শুধু V1-তে।
        # উৎস amp_sig (morph-পরিবার) — P_Amplitude-এর সাথে একই
        # zero-রেফারেন্স, মিশ্র-সিগন্যাল নিয়ম মান্য। ফল lead-median/mean।
        if lead_name == "V1":
            _pt_meas = []
            for _po, _pf in zip(p_onsets, p_offsets):
                if np.isnan(_po) or np.isnan(_pf) or _pf <= _po:
                    continue
                _lo, _hi = int(_po), int(_pf) + 1
                if _hi - _lo < 3:
                    continue
                _pt_meas.append(_p_terminal_measure_v1(amp_sig[_lo:_hi], fs))
            feats["V1_PTFV1_N_Valid_Beats"] = int(len(_pt_meas))
            if _pt_meas:
                feats["V1_P_Terminal_Force_mm_s"] = _safe_median([m["ptfv1_mm_s"] for m in _pt_meas])
                feats["V1_P_Terminal_Neg_Depth_mV"] = _safe_median([m["term_neg_depth_mv"] for m in _pt_meas])
                feats["V1_P_Terminal_Neg_Duration_s"] = _safe_median([m["term_neg_dur_s"] for m in _pt_meas])
                feats["V1_P_Biphasic_Fraction"] = float(np.mean([m["biphasic"] for m in _pt_meas]))
                feats["V1_P_Biphasic_Flag"] = (float(feats["V1_P_Biphasic_Fraction"] >= 0.5)
                                               if len(_pt_meas) >= 3 else np.nan)

        # Q/S amplitude genuine-masked (উপরে মাস্ক তৈরি) —
        # প্রকৃত wave নেই এমন beat-এর arbitrary-baseline মান আর রিপোর্ট হয় না।
        feats[f"{prefix}_Q_Amplitude"] = _safe_median_min_n(q_amp_beats)
        feats[f"{prefix}_S_Amplitude"] = _safe_median_min_n(s_amp_beats)

        # T_Amplitude টেমপ্লেট T-peak থেকে (baseline-সম্পর্কিত, signed) — per-beat
        # t_peak অবস্থান-চঞ্চলতা-নিরপেক্ষ। t_amp_beats শুধু T_Amplitude_Variability-এর
        # জন্য (std-তে ধ্রুবক baseline-offset বাতিল হয়ে যায়)।
        with np.errstate(invalid='ignore'):
            t_amp_beats = _amplitude_at_indices(amp_sig, t_peaks) - pr_baseline_lead
        if tpl_m is not None:
            feats[f"{prefix}_T_Amplitude"] = float(np.clip(tpl_m["T_Amp"], *AMPLITUDE_BOUND_MV))
        else:
            feats[f"{prefix}_T_Amplitude"] = _safe_median_min_n(_apply_bounds(t_amp_beats, *AMPLITUDE_BOUND_MV))

        # Q_R_Ratio: per-beat genuine-Q |Q|/|R|, সাথে R-floor —
        # |R| < 0.50 mV হলে ভাগফল অর্থহীন (NaN)। প্রকৃত Q না থাকলে সেই beat-এ
        # ratio 0 (ভুলভাবে flag হওয়ার কোনো পথ নেই)।
        r_amps_beat = _local_r_amplitude(amp_sig, r_peaks, fs) if n_beats > 0 else np.array([])
        with np.errstate(divide='ignore', invalid='ignore'):
            qr_ratio_beats = np.where(
                r_amps_beat >= MIN_R_FOR_RATIO_MV,
                np.abs(q_amp_beats) / np.abs(r_amps_beat),
                np.nan,
            )
            # q_amp_beats NaN (R-boundary অনুপস্থিত beat) স্বয়ংক্রিয়ভাবে NaN থাকে
        feats[f"{prefix}_Q_R_Ratio"] = _safe_median_min_n(_apply_bounds(qr_ratio_beats, *BOUNDS["Q_R_Ratio"]))

        t_slopes = []
        for p, on in zip(t_peaks, t_onsets):
            if not np.isnan(p) and not np.isnan(on) and p > on:
                amp_diff = amp_sig[int(p)] - amp_sig[int(on)]
                dur_diff = (p - on) / fs
                t_slopes.append(amp_diff / dur_diff if dur_diff > 0 else np.nan)
        feats[f"{prefix}_T_Wave_Slope"] = _safe_median_min_n(t_slopes)

        # RS_Ratio — genuine-S ব্যবহার; S নেই (0.0) অথচ R আছে — এমন
        # beat-এ ভাগফল সংজ্ঞায়িত না (NaN), ভুল সংখ্যার বদলে।
        with np.errstate(divide='ignore', invalid='ignore'):
            s_for_ratio = np.where(s_amp_beats == 0, np.nan, s_amp_beats)
            rs_ratio = np.abs(r_amps_beat / s_for_ratio)
        feats[f"{prefix}_RS_Ratio"] = _safe_median_min_n(_apply_bounds(rs_ratio, *BOUNDS["RS_Ratio"]))

        # ST_Level (টেমপ্লেট J+60ms), ST80_Level (J+80ms), ST_J_Level (J-পয়েন্টে) —
        # সব T-onset−16ms দ্বারা clamped, তাই উচ্চ HR-এও মাপ T-তরঙ্গে পড়ে না।
        # টেমপ্লেট সম্ভব না হলে (rare, <4 beat) honest NaN।
        if tpl_m is not None:
            feats[f"{prefix}_ST_J_Level"] = float(np.clip(tpl_m["ST_J"], *BOUNDS["ST_Deviation"]))
            feats[f"{prefix}_ST_Level"] = float(np.clip(tpl_m["ST60"], *BOUNDS["ST_Deviation"]))
            feats[f"{prefix}_ST80_Level"] = float(np.clip(tpl_m["ST80"], *BOUNDS["ST_Deviation"]))
            feats[f"{prefix}_ST_Slope"] = float(np.clip(tpl_m["ST_Slope"], -50.0, 50.0))

        # Fragmented QRS (fQRS প্রক্সি):
        # (ক) plateau ([+1,0,-1]) sign-chain থেকে বাদ — আগে প্রতিটা extremum
        #     plateau-তে ভেঙে দুবার গোনা হতো;
        # (খ) threshold ≥4 turning points — স্বাভাবিক triphasic qRs কমপ্লেক্সেরই
        #     ৩টি extrema থাকে; আগের `>2` threshold-এ প্রায় সব স্বাভাবিক beat
        #     "fragmented" গোনা হচ্ছিল। এখনও literature-standard fQRS না, proxy।
        n_fragmented_beats = 0
        n_beats_checked = 0
        for lo, hi in zip(qrs_onsets, qrs_offsets):
            if np.isnan(lo) or np.isnan(hi) or hi <= lo:
                continue
            qrs_slice = amp_sig[int(lo):int(hi)]
            if len(qrs_slice) > 4:
                n_beats_checked += 1
                sig_chain = np.sign(np.diff(qrs_slice))
                sig_chain = sig_chain[sig_chain != 0]
                turns = int(np.sum(sig_chain[1:] != sig_chain[:-1])) if len(sig_chain) >= 2 else 0
                if turns >= 4:
                    n_fragmented_beats += 1
        feats[f"{prefix}_Fragmented_QRS_Fraction"] = (
            float(n_fragmented_beats / n_beats_checked) if n_beats_checked >= 3 else np.nan
        )

        # baseline-corrected T amplitude-এরই variability
        valid_t_amps = t_amp_beats[~np.isnan(t_amp_beats)]
        feats[f"{prefix}_T_Amplitude_Variability"] = float(np.std(valid_t_amps, ddof=1)) if len(valid_t_amps) >= 4 else np.nan

    except Exception as e:
        feats[f"{prefix}_Delineate_Error"] = str(e)[:120]

    qt_val = feats[f"{prefix}_QT_Interval"]
    if not pd.isna(qt_val) and not pd.isna(hrv_rr_interval) and hrv_rr_interval > 0:
        feats[f"{prefix}_QTc_Bazett"] = float(qt_val / np.sqrt(hrv_rr_interval))
        feats[f"{prefix}_QTc_Fridericia"] = float(qt_val / (hrv_rr_interval ** (1/3)))
        feats[f"{prefix}_QTc_Framingham"] = float(qt_val + 0.154 * (1.0 - hrv_rr_interval))
    else:
        feats[f"{prefix}_QTc_Bazett"] = np.nan
        feats[f"{prefix}_QTc_Fridericia"] = np.nan
        feats[f"{prefix}_QTc_Framingham"] = np.nan

    return feats


def compute_global_features(all_features):
    p_durs = [all_features.get(f"{lead}_P_Duration", np.nan) for lead in LEAD_NAMES]
    p_durs = [v for v in p_durs if not pd.isna(v)]
    all_features["Global_P_Wave_Dispersion"] = float(max(p_durs) - min(p_durs)) if p_durs else np.nan

    # ---- Frontal plane QRS axis (QRS_Net_Area-ভিত্তিক, I ও aVF) ----
    net_i = all_features.get("I_QRS_Net_Area", np.nan)
    net_avf = all_features.get("aVF_QRS_Net_Area", np.nan)
    if not pd.isna(net_i) and not pd.isna(net_avf) and not (net_i == 0 and net_avf == 0):
        all_features["Frontal_QRS_Axis_Degrees"] = float(np.degrees(np.arctan2(net_avf, net_i)))
    else:
        all_features["Frontal_QRS_Axis_Degrees"] = np.nan

    # ---- Global (multi-lead) QRS Duration ----
    onsets_rel = [all_features.get(f"{lead}_QRS_Onset_Rel_R", np.nan) for lead in LEAD_NAMES]
    offsets_rel = [all_features.get(f"{lead}_QRS_Offset_Rel_R", np.nan) for lead in LEAD_NAMES]
    onsets_rel = [v for v in onsets_rel if not pd.isna(v)]
    offsets_rel = [v for v in offsets_rel if not pd.isna(v)]
    if onsets_rel and offsets_rel:
        global_dur = max(offsets_rel) - min(onsets_rel)
        all_features["Global_QRS_Duration"] = float(global_dur) if 0 < global_dur < 0.30 else np.nan
    else:
        all_features["Global_QRS_Duration"] = np.nan

    # প্রসারিত অফসেট-ভিত্তিক গ্লোবাল প্রস্থ (wide-QRS-রক্ষাকারী সংস্করণ)
    offsets_rel_ext = [all_features.get(f"{lead}_QRS_Offset_Rel_R_Extended", np.nan) for lead in LEAD_NAMES]
    offsets_rel_ext = [v for v in offsets_rel_ext if not pd.isna(v)]
    if onsets_rel and offsets_rel_ext:
        global_dur_ext = max(offsets_rel_ext) - min(onsets_rel)
        all_features["Global_QRS_Duration_Extended"] = (float(global_dur_ext)
                                                        if 0 < global_dur_ext < 0.30 else np.nan)
    else:
        all_features["Global_QRS_Duration_Extended"] = np.nan

    i_qrs = all_features.get("I_R_Amplitude", np.nan)
    ii_qrs = all_features.get("II_R_Amplitude", np.nan)
    if not pd.isna(i_qrs) and not pd.isna(ii_qrs):
        all_features["Global_QRS_Vector_Mag"] = float(np.sqrt(i_qrs**2 + ii_qrs**2))
    else:
        all_features["Global_QRS_Vector_Mag"] = np.nan

    # ---- Lead R-amplitude ratios — denominator floor 0.20 mV ----
    all_features["Ratio_V5_V1"] = _safe_div(all_features.get("V5_R_Amplitude", np.nan), all_features.get("V1_R_Amplitude", np.nan), min_denominator=RATIO_DENOM_FLOOR_MV)
    all_features["Ratio_V6_V1"] = _safe_div(all_features.get("V6_R_Amplitude", np.nan), all_features.get("V1_R_Amplitude", np.nan), min_denominator=RATIO_DENOM_FLOOR_MV)
    all_features["Ratio_II_I"] = _safe_div(all_features.get("II_R_Amplitude", np.nan), all_features.get("I_R_Amplitude", np.nan), min_denominator=RATIO_DENOM_FLOOR_MV)
    all_features["Ratio_aVL_III"] = _safe_div(all_features.get("aVL_R_Amplitude", np.nan), all_features.get("III_R_Amplitude", np.nan), min_denominator=RATIO_DENOM_FLOOR_MV)
    all_features["Ratio_V4_V2"] = _safe_div(all_features.get("V4_R_Amplitude", np.nan), all_features.get("V2_R_Amplitude", np.nan), min_denominator=RATIO_DENOM_FLOOR_MV)
    all_features["Ratio_V6_V3"] = _safe_div(all_features.get("V6_R_Amplitude", np.nan), all_features.get("V3_R_Amplitude", np.nan), min_denominator=RATIO_DENOM_FLOOR_MV)
    all_features["Ratio_aVF_aVR"] = _safe_div(all_features.get("aVF_R_Amplitude", np.nan), all_features.get("aVR_R_Amplitude", np.nan), min_denominator=RATIO_DENOM_FLOOR_MV)
    all_features["Ratio_V3_V1"] = _safe_div(all_features.get("V3_R_Amplitude", np.nan), all_features.get("V1_R_Amplitude", np.nan), min_denominator=RATIO_DENOM_FLOOR_MV)
    all_features["Ratio_V5_V2"] = _safe_div(all_features.get("V5_R_Amplitude", np.nan), all_features.get("V2_R_Amplitude", np.nan), min_denominator=RATIO_DENOM_FLOOR_MV)
    all_features["Ratio_I_III"] = _safe_div(all_features.get("I_R_Amplitude", np.nan), all_features.get("III_R_Amplitude", np.nan), min_denominator=RATIO_DENOM_FLOOR_MV)

    # ---- HYP: standard voltage criteria ----
    s_v1 = all_features.get("V1_S_Amplitude", np.nan)
    r_v5 = all_features.get("V5_R_Amplitude", np.nan)
    r_v6 = all_features.get("V6_R_Amplitude", np.nan)
    r_avl = all_features.get("aVL_R_Amplitude", np.nan)
    s_v3 = all_features.get("V3_S_Amplitude", np.nan)
    r_i = all_features.get("I_R_Amplitude", np.nan)
    s_iii = all_features.get("III_S_Amplitude", np.nan)

    r_v5_v6 = [v for v in [r_v5, r_v6] if not pd.isna(v)]
    if not pd.isna(s_v1) and r_v5_v6:
        all_features["Sokolow_Lyon_Index"] = float(abs(s_v1) + max(r_v5_v6))
    else:
        all_features["Sokolow_Lyon_Index"] = np.nan

    # QRS_Duration_Max Cornell-গণনা থেকে স্বাধীনভাবে গণিত —
    # সবসময় স্বাধীনভাবে গণনা হয়।
    qrs_durs = [all_features.get(f"{lead}_QRS_Duration", np.nan) for lead in LEAD_NAMES]
    qrs_durs = [v for v in qrs_durs if not pd.isna(v)]
    qrs_dur_max = max(qrs_durs) if qrs_durs else np.nan
    all_features["QRS_Duration_Max"] = qrs_dur_max

    if not pd.isna(r_avl) and not pd.isna(s_v3):
        cornell = float(r_avl + abs(s_v3))
        all_features["Cornell_Voltage"] = cornell
        all_features["Cornell_Voltage_Duration_Product"] = (
            float(cornell * qrs_dur_max * 1000) if not pd.isna(qrs_dur_max) else np.nan  # mV*ms এককে
        )
    else:
        all_features["Cornell_Voltage"] = np.nan
        all_features["Cornell_Voltage_Duration_Product"] = np.nan

    if not pd.isna(r_i) and not pd.isna(s_iii):
        all_features["Gubner_Ungerleider_Index"] = float(r_i + abs(s_iii))
    else:
        all_features["Gubner_Ungerleider_Index"] = np.nan

    # ---- MI: precordial R-wave progression (V1->V6) ----
    precordial = ['V1', 'V2', 'V3', 'V4', 'V5', 'V6']
    xs, ys = [], []
    for idx, lead in enumerate(precordial, start=1):
        v = all_features.get(f"{lead}_R_Amplitude", np.nan)
        if not pd.isna(v):
            xs.append(idx)
            ys.append(v)
    if len(xs) >= 3:
        xs_arr, ys_arr = np.array(xs, dtype=float), np.array(ys, dtype=float)
        slope = float(np.polyfit(xs_arr, ys_arr, 1)[0])
        all_features["R_Wave_Progression_Slope"] = slope
    else:
        all_features["R_Wave_Progression_Slope"] = np.nan

    # ---- STTC: ischemia extent — ক্লিনিক্যাল সংজ্ঞা ----
    # Leads_With_ST_Depression: ST80 (J+80ms, হাই-রেটে robust), ≤ -0.10 mV, aVR বাদ
    # Leads_With_T_Inversion: baseline-corrected T, ≤ -0.10 mV, aVR বাদ
    # (aVR-এর reciprocal change স্বাভাবিক ঘটনা, তাই গণনা থেকে বাদ)
    st_levels_80 = [all_features.get(f"{lead}_ST80_Level", np.nan) for lead in LEAD_NAMES if lead not in COUNT_EXCLUDE_LEADS]
    t_amps = [all_features.get(f"{lead}_T_Amplitude", np.nan) for lead in LEAD_NAMES if lead not in COUNT_EXCLUDE_LEADS]
    all_features["Leads_With_ST_Depression"] = int(sum(1 for v in st_levels_80 if not pd.isna(v) and v <= ST_DEPRESSION_THRESHOLD_MV))
    all_features["Leads_With_T_Inversion"] = int(sum(1 for v in t_amps if not pd.isna(v) and v <= T_INVERSION_DEPTH_MV))

    return all_features


def extract_12_lead_features(file_path, ecg_id_hint=None):
    try:
        rec = wfdb.rdrecord(file_path)
        sig_names = getattr(rec, "sig_name", None) or getattr(rec, "sig_names", None)
        signals = np.asarray(rec.p_signal, dtype=float).T   # (n_sig, n_samples)
        fs = float(rec.fs)
        lead_to_idx = _match_lead_indices(sig_names)
        # ইউনিট-গার্ড: PTB-XL verified 'mV'; কোনো কারণে 'uV' পেলে সেই চ্যানেল ×0.001
        for _i, _u in enumerate(rec.units):
            if str(_u).strip().lower() in ("uv", "µv", "microvolt"):
                signals[_i] = signals[_i] * 0.001

        if np.isnan(signals).any() or np.isinf(signals).any():
            return {"ECG_ID": ecg_id_hint, "Ref_Lead": None, "N_RPeaks": 0, "Status": "Invalid Signal (NaN/Inf)"}

        signals_mv = signals * AMPLITUDE_SCALE_CORRECTION

        max_amp = np.max(np.abs(signals_mv))
        if not (PLAUSIBLE_AMPLITUDE_MV_RANGE[0] <= max_amp <= PLAUSIBLE_AMPLITUDE_MV_RANGE[1]):
            return {"ECG_ID": ecg_id_hint, "Ref_Lead": None, "N_RPeaks": 0, "Status": f"Amplitude Scale Suspect (max={max_amp:.4f}mV)"}

        ref_lead, ref_cleaned, r_peaks = _detect_reference_peaks(signals_mv, lead_to_idx, fs)
        if r_peaks is None:
            return {"ECG_ID": ecg_id_hint, "Ref_Lead": None, "N_RPeaks": 0, "Status": "R-Peak Detection Failed"}

        hrv_feats = compute_record_level_hrv(r_peaks, fs)
        hrv_rr_interval = hrv_feats.get("RR_Interval", np.nan)

        all_features = {"ECG_ID": ecg_id_hint, "Ref_Lead": ref_lead, "N_RPeaks": len(r_peaks)}
        for k, v in hrv_feats.items():
            all_features[f"HRV_{k}"] = v

        # HR ক্রস-চেক: median-RR-ভিত্তিক Heart_Rate_BPM বনাম
        # peak-count-ভিত্তিক HR — ব্যবধান বড় হলে double-detection/missed-beat
        # artifact সন্দেহ করে manual পরীক্ষার প্রার্থী (পূর্ণ-রানে mean HR ~102 bpm
        # এসেছিল, প্রত্যাশার চেয়ে বেশি — QC-র জন্য দ্বিতীয় স্বাধীন মাপ)।
        record_duration_s = len(ref_cleaned) / fs if fs else np.nan
        if record_duration_s and record_duration_s > 0:
            all_features["HR_By_Peak_Count"] = float(len(r_peaks) / record_duration_s * 60.0)
        else:
            all_features["HR_By_Peak_Count"] = np.nan

        cleaned_signals_mv_dict = {}
        zero_valid_beat_leads = 0
        for lead in LEAD_NAMES:
            if lead in lead_to_idx:
                if lead == ref_lead:
                    lead_signal = ref_cleaned
                else:
                    lead_signal = nk.ecg_clean(signals_mv[lead_to_idx[lead]], sampling_rate=fs)

                cleaned_signals_mv_dict[lead] = lead_signal
                morph_mv = _wander_removed_mv(signals_mv[lead_to_idx[lead]], fs)  # morph-পরিবার (ST/T ও প্রাবল্য পাথ)
                lead_feats = extract_lead_features(lead_signal, r_peaks, fs, lead, hrv_rr_interval, morph_signal_mv=morph_mv)
                all_features.update(lead_feats)

                if lead_feats.get(f"{lead}_N_Valid_Beats", 0) == 0:
                    zero_valid_beat_leads += 1

        all_features["Leads_With_Zero_Valid_Beats"] = zero_valid_beat_leads

        if 'I' in cleaned_signals_mv_dict and 'II' in cleaned_signals_mv_dict:
            all_features["Corr_I_II"] = safe_corr(cleaned_signals_mv_dict['I'], cleaned_signals_mv_dict['II'])
        else:
            all_features["Corr_I_II"] = np.nan

        if 'V1' in cleaned_signals_mv_dict and 'V2' in cleaned_signals_mv_dict:
            all_features["Corr_V1_V2"] = safe_corr(cleaned_signals_mv_dict['V1'], cleaned_signals_mv_dict['V2'])
        else:
            all_features["Corr_V1_V2"] = np.nan

        if 'V5' in cleaned_signals_mv_dict and 'V6' in cleaned_signals_mv_dict:
            all_features["Corr_V5_V6"] = safe_corr(cleaned_signals_mv_dict['V5'], cleaned_signals_mv_dict['V6'])
        else:
            all_features["Corr_V5_V6"] = np.nan

        all_features = compute_global_features(all_features)
        return all_features
    except Exception as e:
        return {"ECG_ID": ecg_id_hint, "Ref_Lead": None, "N_RPeaks": 0, "Status": f"Error: {str(e)}"}


# ============================================================
# মেটাডেটা প্রসেসিং, BMI এবং ফাইনাল লেবেলিং
# ============================================================

def add_disease_and_global_features(df, metadata_path):
    df_meta = pd.read_csv(metadata_path)

    df_meta['ecg_id'] = df_meta['ecg_id'].astype(str).str.replace(r'\D', '', regex=True).astype(int)
    df['ECG_ID'] = df['ECG_ID'].astype(str).str.replace(r'\D', '', regex=True).astype(int)

    merge_cols = ["ecg_id", "primary_label"]
    if "height" in df_meta.columns and "weight" in df_meta.columns:
        merge_cols.extend(["height", "weight"])

    df = pd.merge(df, df_meta[merge_cols], left_on="ECG_ID", right_on="ecg_id", how="left")
    df = df.drop(columns=["ecg_id"])

    if "height" in df.columns and "weight" in df.columns:
        with np.errstate(divide='ignore', invalid='ignore'):
            df["BMI"] = df["weight"] / ((df["height"] / 100.0) ** 2)
        df["BMI"] = np.where(np.isinf(df["BMI"]) | np.isnan(df["BMI"]) | (df["BMI"] <= 0), np.nan, df["BMI"])
        df = df.drop(columns=["height", "weight"])
    else:
        df["BMI"] = np.nan

    df["Disease_Name"] = df["primary_label"].astype(str).str.strip()
    df = df[df["Disease_Name"].isin(DISEASE_CLASSES)].reset_index(drop=True)

    le = LabelEncoder()
    le.fit(DISEASE_CLASSES)
    df["Disease_Label_Encoded"] = le.transform(df["Disease_Name"])

    for cls in DISEASE_CLASSES:
        df[f"Disease_{cls}"] = (df["Disease_Name"] == cls).astype(int)

    df = df.drop(columns=["primary_label"])

    if "HRV_RR_Interval" in df.columns:
        with np.errstate(divide='ignore', invalid='ignore'):
            df["Heart_Rate_BPM"] = 60.0 / df["HRV_RR_Interval"]
        df["Heart_Rate_BPM"] = np.where(
            (df["Heart_Rate_BPM"] < 20) | (df["Heart_Rate_BPM"] > 250),
            np.nan,
            df["Heart_Rate_BPM"]
        )
    else:
        df["Heart_Rate_BPM"] = np.nan

    return df


# ============================================================
# এক্সট্র্যাকশন ও পাইপলাইন রানার ইঞ্জিন
# ============================================================

def ensure_extracted(rar_path, extract_dir):
    wfdb_folder = os.path.join(extract_dir, "records500")
    if os.path.exists(wfdb_folder) and len(os.listdir(wfdb_folder)) > 0:
        return True
    if os.path.exists(extract_dir):
        shutil.rmtree(extract_dir)
    os.makedirs(extract_dir)
    try:
        patoolib.extract_archive(rar_path, outdir=extract_dir)
    except Exception:
        return False
    return True


def record_path_from_id(id_or_filename, extract_dir):
    """'00001' বা 1 → extracted_wfdb/records500/00000/00001_hr"""
    import re as _re
    stem = _re.sub(r"\D", "", str(id_or_filename))
    n = int(stem)
    return os.path.join(extract_dir, "records500", f"{(n // 1000) * 1000:05d}", f"{n:05d}_hr")


def process_single_ecg(row_dict, extract_dir):
    hint = row_dict.get('ecg_id') or row_dict.get('edf_filename')
    rec_file = record_path_from_id(hint, extract_dir)
    if os.path.exists(rec_file + ".dat"):
        return extract_12_lead_features(rec_file, ecg_id_hint=row_dict.get('ecg_id'))
    return {"ECG_ID": row_dict.get('ecg_id'), "Ref_Lead": None, "N_RPeaks": 0, "Status": "File Not Found"}


def get_already_processed_ids(output_path):
    if not os.path.exists(output_path):
        return set()
    try:
        df_existing = pd.read_csv(output_path, usecols=["ECG_ID"])
        return set(df_existing["ECG_ID"].dropna().astype(str).str.replace(r'\D', '', regex=True).astype(int).unique().tolist())
    except Exception:
        return set()


def build_canonical_columns(metadata_path, extract_dir):
    df_meta = pd.read_csv(metadata_path)
    for _, row in df_meta.iterrows():
        res = process_single_ecg(row.to_dict(), extract_dir)
        if res and res.get("Status") is None and len(res) > 10:
            cols = list(res.keys())
            if "Status" not in cols:
                cols.append("Status")
            return cols
    return ["ECG_ID", "Ref_Lead", "N_RPeaks", "Status"]


def run_extraction(rar_path, metadata_path, output_path, extract_dir=EXTRACT_DIR, save_every_n=SAVE_EVERY_N, max_workers=None):
    if not ensure_extracted(rar_path, extract_dir):
        return

    df_meta = pd.read_csv(metadata_path)
    processed_ids = get_already_processed_ids(output_path)

    if processed_ids:
        df_meta["match_id"] = df_meta["ecg_id"].astype(str).str.replace(r'\D', '', regex=True).astype(int)
        before = len(df_meta)
        df_meta = df_meta[~df_meta["match_id"].isin(processed_ids)]
        df_meta = df_meta.drop(columns=["match_id"])
        print(f"➡️ {before} টির মধ্যে {before - len(df_meta)} টি আগে প্রসেস হয়েছে। বাকি {len(df_meta)} টি হবে।")
    else:
        print(f"➡️ নতুন শুরু। মোট {len(df_meta)} টি ফাইল প্রসেস হবে।")

    if len(df_meta) == 0:
        return

    canonical_columns = build_canonical_columns(metadata_path, extract_dir)
    rows_list = [row.to_dict() for _, row in df_meta.iterrows()]
    results = []
    n_failed_total = 0
    workers = max_workers if max_workers else os.cpu_count()

    def _flush(results):
        nonlocal n_failed_total
        if not results:
            return
        df_temp = pd.DataFrame(results)
        if "Status" in df_temp.columns:
            n_failed_total += int(df_temp["Status"].notna().sum())
        df_temp = df_temp.reindex(columns=canonical_columns)
        df_temp.to_csv(output_path, mode='a', header=not os.path.exists(output_path), index=False)

    with concurrent.futures.ProcessPoolExecutor(max_workers=workers) as executor:
        futures = {executor.submit(process_single_ecg, row, extract_dir): row for row in rows_list}
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Processing"):
            try:
                res = future.result()
            except Exception as e:
                row_info = futures[future]
                res = {"ECG_ID": row_info.get('ecg_id'), "Ref_Lead": None, "N_RPeaks": 0, "Status": f"Fatal Worker Error: {str(e)}"}
            if res:
                results.append(res)
            if len(results) >= save_every_n:
                _flush(results)
                results = []
    _flush(results)

    print(f"\n📊 এই রানে মোট fail হওয়া রেকর্ড: {n_failed_total}")


def finalize_features(output_path, metadata_path, final_output_path):
    df = pd.read_csv(output_path)
    if "Status" in df.columns:
        n_failed = int(df["Status"].notna().sum())
        failed_ids = df.loc[df["Status"].notna(), "ECG_ID"].tolist()
        if n_failed > 0:
            print(f"⚠️ মোট {n_failed} টি ফাইল ফিচার এক্সট্র্যাকশনে ব্যর্থ হয়েছে এবং ফাইনাল ফাইল থেকে বাদ পড়েছে।")
            print(f"   বাদ পড়া ECG_ID (প্রথম ২০টি): {failed_ids[:20]}")
        df = df[df["Status"].isna()].drop(columns=["Status"])

    df["ECG_ID"] = df["ECG_ID"].astype(str).str.replace(r'\D', '', regex=True).astype(int)
    df = df.drop_duplicates(subset=["ECG_ID"], keep="first")
    df = add_disease_and_global_features(df, metadata_path)
    df = df.sort_values("ECG_ID").reset_index(drop=True)
    df.to_csv(final_output_path, index=False)
    print(f"\n✅ চূড়ান্ত ফাইল সেভ হয়েছে: {final_output_path}")
    print(f"✅ মোট রেকর্ড সংখ্যা: {len(df)} / প্রত্যাশিত 20970")
    print(f"✅ মোট কলাম সংখ্যা: {df.shape[1]}")


# =============================================================================
# ONE-STEP USER CONFIGURATION
# =============================================================================
BASE_DIR = os.environ.get(
    "PTBXL_BASE_DIR",
    "/content/drive/MyDrive/PTB-XL Dataset Analysis"
)

# Required inputs already used by the original pipeline.
RAR_FILE_PATH_ONE_STEP = os.path.join(BASE_DIR, "records500.rar")
METADATA_CSV_ONE_STEP = os.path.join(
    BASE_DIR, "ptbxl_primary_NORM_MI_STTC_HYP_CD_labeled_metadata.csv"
)
PTBXL_DATABASE_CSV = os.path.join(BASE_DIR, "ptbxl_database.csv")

# Intermediate outputs are retained to support resume after Colab interruption.
MASTER_RAW_CSV = os.path.join(BASE_DIR, "ECG_Master_Features_Fixed_File.csv")
MASTER_FINAL_CSV = MASTER_RAW_CSV.replace(".csv", "_FINAL.csv")
MODEL_READY_CSV = os.path.join(BASE_DIR, "ECG_Master_Features_MODEL_ready.csv")
FQRS_CSV_ONE_STEP = os.path.join(BASE_DIR, "ECG_fQRS_Features.csv")
FINAL_OUTPUT_CSV = os.path.join(BASE_DIR, "ECG_MODEL_ready_FINAL_with_StratFold.csv")
AUDIT_JSON = os.path.join(BASE_DIR, "ECG_MODEL_ready_FINAL_with_StratFold_AUDIT.json")

# Local extraction is faster than reading every waveform from Drive.
EXTRACT_DIR_ONE_STEP = "/content/extracted_wfdb" if os.path.isdir("/content") else os.path.join(BASE_DIR, "extracted_wfdb")

# False = safely resume existing intermediate CSVs. True = delete generated
# intermediate/output CSVs and rebuild from scratch with this exact code.
CLEAN_REBUILD = False
MAX_WORKERS = None
EXPECTED_FINAL_ROWS = 20310
EXPECTED_FINAL_COLUMNS = 555
# Optional: path to a previous final file for exact schema/value comparison.
# Keep as None if unavailable, or set to a differently named reference CSV.
REFERENCE_FINAL_CSV = None

for required in [RAR_FILE_PATH_ONE_STEP, METADATA_CSV_ONE_STEP, PTBXL_DATABASE_CSV]:
    if not os.path.exists(required):
        raise FileNotFoundError(f"Required input not found: {required}")

meta_check = pd.read_csv(METADATA_CSV_ONE_STEP, nrows=5)
required_meta_cols = {"ecg_id", "primary_label"}
if not required_meta_cols.issubset(meta_check.columns):
    raise ValueError(
        f"Metadata must contain {sorted(required_meta_cols)}; got {list(meta_check.columns)}"
    )

if CLEAN_REBUILD:
    for p in [MASTER_RAW_CSV, MASTER_FINAL_CSV, MODEL_READY_CSV,
              FQRS_CSV_ONE_STEP, FINAL_OUTPUT_CSV, AUDIT_JSON]:
        if os.path.exists(p):
            os.remove(p)
            print(f"Removed old generated file: {p}")

# Override the original notebook globals while preserving all extraction logic.
RAR_FILE_PATH = RAR_FILE_PATH_ONE_STEP
METADATA_CSV = METADATA_CSV_ONE_STEP
OUTPUT_CSV = MASTER_RAW_CSV
EXTRACT_DIR = EXTRACT_DIR_ONE_STEP


# =============================================================================
# STAGE A — ORIGINAL FIXED-FILE FEATURE EXTRACTION (Cell 41)
# =============================================================================
print("\n[1/5] Running/resuming the final fixed-file feature extractor...")
run_extraction(
    RAR_FILE_PATH_ONE_STEP,
    METADATA_CSV_ONE_STEP,
    MASTER_RAW_CSV,
    extract_dir=EXTRACT_DIR_ONE_STEP,
    save_every_n=SAVE_EVERY_N,
    max_workers=MAX_WORKERS,
)

_meta_full = pd.read_csv(METADATA_CSV_ONE_STEP)
_processed = get_already_processed_ids(MASTER_RAW_CSV)
if len(_processed) < len(_meta_full):
    raise RuntimeError(
        f"Extraction incomplete: {len(_processed)}/{len(_meta_full)} metadata records processed. "
        "Rerun the same script to resume."
    )

finalize_features(MASTER_RAW_CSV, METADATA_CSV_ONE_STEP, MASTER_FINAL_CSV)

# =============================================================================
# STAGE B — EXACT MODEL-READY FILTER/DROP LOGIC (original Cell 49)
# =============================================================================
print("\n[2/5] Applying the original model-ready QC and column-removal stage...")
LEAD_NAMES_ONE_STEP = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF',
                       'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
df_model = pd.read_csv(MASTER_FINAL_CSV)
_n_before_hr = len(df_model)

# Original HRV forensic record filter.
mask_ok = (
    df_model["Heart_Rate_BPM"] - df_model["HR_By_Peak_Count"]
).abs() <= 10
df_model = df_model.loc[mask_ok].reset_index(drop=True)
_n_hr_removed = int(_n_before_hr - len(df_model))

# Original QC-determined drop list.
drop_cols = (
    [f"{L}_Delineate_Error" for L in LEAD_NAMES_ONE_STEP] +
    [f"{L}_Fragmented_QRS_Fraction" for L in LEAD_NAMES_ONE_STEP] +
    ["Leads_With_Zero_Valid_Beats", "Global_Min_Lead_Valid_Fraction",
     "Global_QRS_Activation_Window",
     "Global_QRS_Duration", "Global_QRS_Duration_Extended",
     "QRS_Duration_Max", "Ref_Lead", "HR_By_Peak_Count", "BMI"]
)
_dropped_present = [c for c in drop_cols if c in df_model.columns]
df_model = df_model.drop(columns=_dropped_present)

# Original corrected Cornell voltage-duration product.
if ("Cornell_Voltage" in df_model.columns and
        "Global_QRS_Duration_MedianPerLead_Extended" in df_model.columns):
    df_model["Cornell_Voltage_Duration_Product_Clean"] = (
        df_model["Cornell_Voltage"] *
        df_model["Global_QRS_Duration_MedianPerLead_Extended"] * 1000
    )
    if "Cornell_Voltage_Duration_Product" in df_model.columns:
        df_model = df_model.drop(columns=["Cornell_Voltage_Duration_Product"])

df_model.to_csv(MODEL_READY_CSV, index=False)
print(f"Model-ready intermediate: {df_model.shape} -> {MODEL_READY_CSV}")

# Make the exact fixed-file pipeline globals available to the original fQRS cell.
METADATA_CSV = METADATA_CSV_ONE_STEP
EXTRACT_DIR = EXTRACT_DIR_ONE_STEP
FQRS_RUN = True


# =============================================================================
# STAGE C — ORIGINAL fQRS ADD-ON (Cell 52)
# =============================================================================
print("\n[3/5] Running/resuming the original fQRS add-on...")
# ============================================================
# Fragmented QRS (fQRS) ফিচার অ্যাড-অন — ডাস (Das) ক্লিনিক্যাল সংজ্ঞা-ভিত্তিক
#
# পূর্বশর্ত: একই Colab সেশনে fixed_file.py-এর পুরো কোড আগে রান করা
# (সব ফাংশন/কনস্ট্যান্ট মেমরিতে); প্রথমবার হলে →  !pip -q install wfdb
#
# পদ্ধতি (সাহিত্য-সংজ্ঞার গণনীয় রূপাত্ম):
#   • সিগন্যাল: 0.15–100 Hz ব্যান্ডপাস (ডাস পরীক্ষণ-সেটিং), শারীরিক একক mV।
#   • প্রতি লিডে R-কেন্দ্রিক median-beat টেমপ্লেট — নয়েজ গড়-প্রভাবে কমে,
#     বিচারের বস্তু হয় প্রতিনিধি QRS-আকৃতি (দৃশ্য-পাঠের সমসম্মান ধারণা)।
#   • প্রতি-বীট উইন্ডো-সূচনা prominence delineation থেকে (৬০ ms q/s-সীমা-ক্যাপসহ);
#     অফসেট-প্রান্তে অতিরিক্ত সুরক্ষা — delineation মাঝে মাঝে টার্মিনাল R′/slur
#     কেটে ফেলে, তাই টেমপ্লেটের ওপর ডেরিভেটিভ-ভিত্তিক প্রসারিত অফসেটের সাথে
#     তুলনা করে বড়টি নেওয়া হয় (পাইপলাইনের যাচাইকৃত একই যন্ত্রাণী)।
#   • ডিফ্লেকশন গণনা: তাৎপর্যপূর্ণ extremum = prominence ≥ max(0.04 mV,
#     5%×QRS-peak-to-peak); ১০ ms-এর কাছাকাছি জোড়া মিলিত হয় (slur-merge)।
#   • লিড-ফ্ল্যাগ (ডাস narrow-সংজ্ঞার গণনীয় সংগতি):
#       fQRSলিড = RSR′-প্যাটার্ন (≥২ তাৎপর্যপূর্ণ ধনাত্মক শিখর, মাঝখানে
#       তাৎপর্যপূর্ণ নিমজ্জন)  অথবা  মোট ≥৪ তাৎপর্যপূর্ণ extremum
#       (নর্মাল qRs = ৩ — তাই ৪ মানেই অতিরিক্ত ডিফ্লেকশন)।
#   • রেকর্ড-সংজ্ঞা: শুধু সরু-QRS রেকর্ডে (cross-lead মিডিয়ান প্রস্থ <120 ms;
#     প্রশস্ত-QRS-এ আলাদা f-wQRS মানদণ্ড প্রযোজ্য — বিস্তারিত limitation)।
#     ভূখণ্ড ≥২ লিডে ফ্ল্যাগ → fQRS_Territory_Flag=1 (ভূখণ্ড — ডাস-রেফারেন্স
#     অনুযায়ী: inferior II/III/aVF, lateral I/aVL/V5/V6, anterior V1–V4)।
#
# সৎ সীমাবদ্ধতা (পেপারে উল্লেখ্য):
#   ১) প্রশস্ত-QRS (BBB ক্লাস) রেকর্ডে রেকর্ড-লেভেল ফ্ল্যাগ দেওয়া হয় না
#      (NaN) — per-lead সংখ্যা তবু রিপোর্ট হয় (f-wQRS মানদণ্ড ভবিষ্যৎ কাজ)।
#   ২) নিম্ন-ভোল্টেজ লিড (QRS peak-to-peak <0.15 mV) "মূল্যায়ন অসম্ভব" →
#      honest NaN — নয়েজে ভরা সংখ্যা রিপোর্ট না করাই নীতি।
#   ৩) ভূখণ্ডে "≥২ contiguous" প্রয়োগে ভূখণ্ড-সদস্যতা-ভিত্তিক কনভেনশন
#      (পাইপলাইনের pathological-Q-র রেফারেন্স-কাঠামোর সাথে সামঞ্জস্য)।
#
# আউটপুট: আলাদা CSV (ECG_ID-কী) — মূল ফিচার-ফাইল অক্ষত থাকে; শেষে merge
# নির্দেশনা ছাপে। চেকপয়েন্ট প্রতি ৫০০ রেকর্ডে; ব্যাহত হলে রিজিউম হয়।
# পাইলট-মোড (FQRS_PILOT_N সেট থাকলে): CSV লেখে না — শুধু শ্রেণিভেদে
# ক্লিনিক্যাল গ্রেডিয়েন্ট ছাপে (ফুল রানের পূর্ব-যাচাই)।
#!pip -q install "numpy==2.0.2" "pandas==2.2.2" "neurokit2==0.2.13" wfdb
# ============================================================

import os
import numpy as np
import pandas as pd
import wfdb
import neurokit2 as nk
from scipy.signal import butter, sosfiltfilt, find_peaks
FQRS_PILOT_N = None

# ------------------------------------------------------------
# নির্ভরতা-গার্ড: fixed_file.py-এর সংজ্ঞা সেশনে আছে কিনা
# ------------------------------------------------------------
if "METADATA_CSV" not in globals():
    METADATA_CSV = "/content/drive/MyDrive/PTB-XL Dataset Analysis/ptbxl_primary_NORM_MI_STTC_HYP_CD_labeled_metadata.csv"
if "EXTRACT_DIR" not in globals():
    EXTRACT_DIR = "extracted_wfdb"

_need_consts = [n for n in ("LEAD_NAMES", "DELINEATE_METHOD", "DELINEATE_KWARGS",
                            "record_path_from_id", "MAX_QS_EXTENSION_MS")
                if n not in globals()]
# হেল্পার ফাংশনের নাম আন্ডারস্কোর-সহ/ছাড়া — দুই রীতিতেই খোঁজা হয়,
# যে-কপি দিয়েই মূল এক্সট্রাকশন করা হোক এই অ্যাড-অন কাজ করবে
_fn_detect = globals().get("_detect_reference_peaks") or globals().get("detect_reference_peaks")
_fn_template = globals().get("_build_beat_template") or globals().get("build_beat_template")
_fn_extoff = globals().get("_extended_qrs_offset") or globals().get("extended_qrs_offset")
_missing_fns = [n for n, f in (("detect_reference_peaks", _fn_detect),
                               ("build_beat_template", _fn_template),
                               ("extended_qrs_offset", _fn_extoff)) if f is None]
if _need_consts or _missing_fns:
    raise SystemExit(
        f"❌ মূল পাইপলাইনের সংজ্ঞা এই সেশনে নেই (অনুপস্থিত: {_need_consts + _missing_fns})।\n"
        "   সঠিক ক্রম:\n"
        "   (১) আগে মূল এক্সট্রাকশন-কোড (যে-কপি দিয়েই ফিচার ফাইল তৈরি হয়েছে)\n"
        "       পুরোটা এক সেলে রান করো; এক্সট্রাকশন শুরু/শেষ হলে একবার Stop চাপো\n"
        "       — সংজ্ঞা মেমরিতে থাকে।\n"
        "   (২) তারপর এই সেল রান করো।\n"
        "   দ্রষ্টব্য: Colab রানটাইম রিস্টার্ট হলে সব সংজ্ঞা হারায় — তখন (১) আবার করতে হবে।")

_ver = globals().get("PIPELINE_ID", "অজানা")
print(f"✓ পাইপলাইন-সংজ্ঞা পাওয়া গেছে (PIPELINE_ID = {_ver}) — fQRS অ্যাড-অন প্রস্তুত।")

# ------------------------------------------------------------
# কনফিগারেশন
# ------------------------------------------------------------
FQRS_OUTPUT_CSV = FQRS_CSV_ONE_STEP
FQRS_BANDPASS_HZ = (0.15, 100.0)     # ডাস পরীক্ষণ-সেটিং
FQRS_PROM_ABS_MV = 0.04              # তাৎপর্যপূর্ণতার ন্যূনতম প্রাবল্য-ফ্লোর
FQRS_PROM_FRAC = 0.05                # × QRS peak-to-peak
FQRS_REFRACT_MS = 10.0               # slur-merge রিফ্র্যাক্টরি
FQRS_MIN_P2P_MV = 0.15               # এর নিচে QRS "মূল্যায়ন অসম্ভব"
FQRS_NARROW_MAX_S = 0.120            # রেকর্ড-লেভেল সংজ্ঞা শুধু সরু-QRS-এ
FQRS_TERRITORIES = {                 # ডাস-ভূখণ্ড রেফারেন্স
    "inferior": ["II", "III", "aVF"],
    "lateral": ["I", "aVL", "V5", "V6"],
    "anterior": ["V1", "V2", "V3", "V4"],
}
_SAVE_EVERY = 500

# ------------------------------------------------------------
# কোর ফাংশন
# ------------------------------------------------------------

def _fqrs_bandpass(sig_mv, fs):
    nyq = fs / 2.0
    lo, hi = FQRS_BANDPASS_HZ[0] / nyq, min(FQRS_BANDPASS_HZ[1] / nyq, 0.99)
    sos = butter(2, [lo, hi], btype="bandpass", output="sos")
    return sosfiltfilt(sos, np.asarray(sig_mv, dtype=float))


def _qrs_deflections(seg, fs):
    """টেমপ্লেট QRS-সেগমেন্টে ডাস-ক্যালিবার ডিফ্লেকশন গণনা।
    রিটার্ন: (n_extrema, rsr_pattern) — বা (None, None) মূল্যায়ন অসম্ভব হলে।"""
    seg = np.asarray(seg, dtype=float)
    if seg.size < int(round(0.030 * fs)):
        return None, None
    p2p = float(np.ptp(seg))
    if p2p < FQRS_MIN_P2P_MV:
        return None, None
    thr = max(FQRS_PROM_ABS_MV, FQRS_PROM_FRAC * p2p)
    pos_pk, pos_pr = find_peaks(seg, prominence=thr)
    neg_pk, neg_pr = find_peaks(-seg, prominence=thr)
    ents = [(int(i), 1, float(p)) for i, p in zip(pos_pk, pos_pr["prominences"])]
    ents += [(int(i), -1, float(p)) for i, p in zip(neg_pk, neg_pr["prominences"])]
    ents.sort()
    # refractory collapse: কাছাকাছি জোড়া extremum → বড় prominence-এটিই রাখো
    refr = max(1, int(round(FQRS_REFRACT_MS / 1000.0 * fs)))
    merged = []
    for e in ents:
        if merged and e[0] - merged[-1][0] < refr:
            if e[2] > merged[-1][2]:
                merged[-1] = e
        else:
            merged.append(list(e) if isinstance(e, tuple) else e)
    merged = [tuple(m) for m in merged]
    n_ext = len(merged)
    pos_idx = [i for i, s, _ in merged if s > 0]
    neg_idx = [i for i, s, _ in merged if s < 0]
    rsr = False
    if len(pos_idx) >= 2 and neg_idx:
        for a, b in zip(pos_idx[:-1], pos_idx[1:]):
            if any(a < n < b for n in neg_idx):
                rsr = True
                break
    return n_ext, rsr


def _per_beat_vals(waves, key, n_beats):
    vals = np.asarray(waves.get(key, []), dtype=float)
    if len(vals) < n_beats:
        vals = np.concatenate([vals, np.full(n_beats - len(vals), np.nan)])
    return vals[:n_beats]


def compute_fqrs_record(file_path, ecg_id_hint=None):
    """একটি রেকর্ডের সম্পূর্ণ fQRS পরিমাপ। রিটার্ন: ফিচার-dict (বা Status-সহ error)।"""
    leads = LEAD_NAMES
    try:
        rec = wfdb.rdrecord(file_path)
        sig_names = getattr(rec, "sig_name", None) or getattr(rec, "sig_names", None)
        signals = np.asarray(rec.p_signal, dtype=float).T
        fs = float(rec.fs)
        lead_to_idx = {n.strip().upper(): i for i, n in enumerate(sig_names)}
        lead_to_idx = {L: lead_to_idx[L.upper()] for L in leads if L.upper() in lead_to_idx}
        for _i, _u in enumerate(rec.units):
            if str(_u).strip().lower() in ("uv", "µv", "microvolt"):
                signals[_i] = signals[_i] * 0.001
        if np.isnan(signals).any() or np.isinf(signals).any():
            return {"ECG_ID": ecg_id_hint, "Status": "Invalid Signal (NaN/Inf)"}

        ref_lead, _, r_peaks = _fn_detect(signals, lead_to_idx, fs)
        if r_peaks is None:
            return {"ECG_ID": ecg_id_hint, "Status": "R-Peak Detection Failed"}
        n_beats = len(r_peaks)

        feats = {"ECG_ID": ecg_id_hint}
        max_ext = int(round(MAX_QS_EXTENSION_MS / 1000.0 * fs))   # 60 ms ক্যাপ (পাইপলাইন-সংগত)
        widths = []
        for lead in leads:
            c_ext, c_flag = f"{lead}_fQRS_Extrema_Count", f"{lead}_fQRS_Flag"
            feats[c_ext], feats[c_flag] = np.nan, np.nan
            if lead not in lead_to_idx:
                continue
            raw = signals[lead_to_idx[lead]]
            try:
                filt = _fqrs_bandpass(raw, fs)
                tpl, r_idx, _rr = _fn_template(filt, r_peaks, fs)
                if tpl is None:
                    continue
                cleaned = nk.ecg_clean(raw, sampling_rate=fs)
                _, waves = nk.ecg_delineate(cleaned, r_peaks, sampling_rate=fs,
                                            method=DELINEATE_METHOD, **DELINEATE_KWARGS)
                r_on = _per_beat_vals(waves, "ECG_R_Onsets", n_beats)
                r_off = _per_beat_vals(waves, "ECG_R_Offsets", n_beats)
                q_pk = _per_beat_vals(waves, "ECG_Q_Peaks", n_beats)
                s_pk = _per_beat_vals(waves, "ECG_S_Peaks", n_beats)
                with np.errstate(invalid="ignore"):
                    onsets = np.where((~np.isnan(r_on)) & (~np.isnan(q_pk))
                                      & (r_on - q_pk >= 0) & (r_on - q_pk <= max_ext), q_pk, r_on)
                    offsets = np.where((~np.isnan(r_off)) & (~np.isnan(s_pk))
                                       & (s_pk - r_off >= 0) & (s_pk - r_off <= max_ext), s_pk, r_off)
                rel_on = (onsets - r_peaks) / fs
                rel_off = (offsets - r_peaks) / fs
                rel_on = rel_on[~np.isnan(rel_on)]
                rel_off = rel_off[~np.isnan(rel_off)]
                if len(rel_on) < 3 or len(rel_off) < 3:
                    continue
                on_med, off_med = float(np.median(rel_on)), float(np.median(rel_off))
                # অফসেট-সুরক্ষা: টার্মিনাল R′/slur delineation-উইন্ডোর বাইরে পড়ে
                # গেলে টেমপ্লেট-ভিত্তিক প্রসারিত অফসেট পূর্ণ QRS ধরে — বড়টি নেই
                _ext_end = min(int(r_idx + round(0.200 * fs)), len(tpl) - 2)
                _ext_off = _fn_extoff(tpl, r_idx, _ext_end, fs)
                if _ext_off is not None:
                    off_med = max(off_med, (_ext_off - r_idx) / fs)
                widths.append(off_med - on_med)
                pad = int(round(0.006 * fs))
                lo = int(r_idx + round(on_med * fs)) - pad
                hi = int(r_idx + round(off_med * fs)) + pad
                lo, hi = max(3, lo), min(len(tpl) - 3, hi)
                if hi - lo < int(round(0.030 * fs)):
                    continue
                n_ext, rsr = _qrs_deflections(tpl[lo:hi], fs)
                if n_ext is None:
                    continue
                feats[c_ext] = int(n_ext)
                feats[c_flag] = float((n_ext >= 4) or bool(rsr))
            except Exception:
                continue

        flags = [feats.get(f"{L}_fQRS_Flag", np.nan) for L in leads]
        valid_flags = [f for f in flags if not pd.isna(f)]
        feats["fQRS_Lead_Count"] = int(sum(int(f) for f in valid_flags)) if valid_flags else np.nan
        narrow = (float(np.median(widths)) < FQRS_NARROW_MAX_S) if widths else np.nan
        feats["fQRS_Narrow_QRS"] = (float(narrow) if not pd.isna(narrow) else np.nan)
        # রেকর্ড-লেভেল ভূখণ্ড-সংজ্ঞা: শুধু সরু-QRS রেকর্ডে
        if narrow is True:
            terr_hit = False
            for terr_leads in FQRS_TERRITORIES.values():
                hits = sum(1 for L in terr_leads if feats.get(f"{L}_fQRS_Flag", 0) == 1.0)
                if hits >= 2:
                    terr_hit = True
                    break
            feats["fQRS_Territory_Flag"] = float(terr_hit)
        else:
            feats["fQRS_Territory_Flag"] = np.nan
        return feats
    except Exception as e:
        return {"ECG_ID": ecg_id_hint, "Status": f"Error: {str(e)[:120]}"}


def _fqrs_path_from_id(id_or_filename, extract_dir):
    return record_path_from_id(id_or_filename, extract_dir)


def _fqrs_process_row(row_dict, extract_dir):
    hint = row_dict.get("ecg_id") or row_dict.get("edf_filename")
    rec_file = _fqrs_path_from_id(hint, extract_dir)
    if os.path.exists(rec_file + ".dat"):
        return compute_fqrs_record(rec_file, ecg_id_hint=row_dict.get("ecg_id"))
    return {"ECG_ID": row_dict.get("ecg_id"), "Status": "File Not Found"}


# ------------------------------------------------------------
# রানার (Colab সেলে পেস্ট করলে চলে; FQRS_RUN=False দিলে শুধু সংজ্ঞা লোড হয়)
# ------------------------------------------------------------
if globals().get("FQRS_RUN", True):
    import concurrent.futures
    from tqdm import tqdm as _tqdm_fq

    df_meta_fq = pd.read_csv(METADATA_CSV)
    _pilot_n = globals().get("FQRS_PILOT_N", None)
    if _pilot_n:
        df_meta_fq = df_meta_fq.head(int(_pilot_n))
        print(f"➡️ পাইলট-মোড: {len(df_meta_fq)} টি রেকর্ডে ক্লিনিক্যাল গ্রেডিয়েন্ট যাচাই (CSV লেখা হবে না)\n")
    else:
        ensure_fn = globals().get("ensure_extracted")
        if ensure_fn and not ensure_fn(globals().get("RAR_FILE_PATH", ""), EXTRACT_DIR):
            raise SystemExit("❌ WFDB এক্সট্রাকশন পাওয়া যাচ্ছে না।")
        done_ids = set()
        if os.path.exists(FQRS_OUTPUT_CSV):
            try:
                _prev = pd.read_csv(FQRS_OUTPUT_CSV, usecols=["ECG_ID"])
                done_ids = set(_prev["ECG_ID"].dropna().astype(str).str.replace(r"\D", "", regex=True).astype(int))
            except Exception:
                done_ids = set()
        if done_ids:
            df_meta_fq["match_id"] = df_meta_fq["ecg_id"].astype(str).str.replace(r"\D", "", regex=True).astype(int)
            df_meta_fq = df_meta_fq[~df_meta_fq["match_id"].isin(done_ids)].drop(columns=["match_id"])
            print(f"➡️ রিজিউম: {len(done_ids)} টি আগে হয়েছে; বাকি {len(df_meta_fq)} টি।")
        else:
            print(f"➡️ ফুল-রান: মোট {len(df_meta_fq)} টি রেকর্ড।")

    if len(df_meta_fq) > 0:
        _rows_fq = [r.to_dict() for _, r in df_meta_fq.iterrows()]
        _results_fq = []
        _wq = globals().get("max_workers") or os.cpu_count()
        import concurrent.futures as _cf
        with _cf.ProcessPoolExecutor(max_workers=_wq) as _ex:
            _futs = {_ex.submit(_fqrs_process_row, r, EXTRACT_DIR): r for r in _rows_fq}
            for _f in _tqdm_fq(_cf.as_completed(_futs), total=len(_futs), desc="fQRS"):
                try:
                    _r = _f.result()
                except Exception as _e:
                    _r = {"ECG_ID": _futs[_f].get("ecg_id"), "Status": f"Worker Error: {str(_e)[:100]}"}
                if _r:
                    _results_fq.append(_r)
                if not _pilot_n and len(_results_fq) >= _SAVE_EVERY:
                    pd.DataFrame(_results_fq).to_csv(FQRS_OUTPUT_CSV, mode="a",
                                                     header=not os.path.exists(FQRS_OUTPUT_CSV), index=False)
                    _results_fq = []
        if not _pilot_n and _results_fq:
            pd.DataFrame(_results_fq).to_csv(FQRS_OUTPUT_CSV, mode="a",
                                             header=not os.path.exists(FQRS_OUTPUT_CSV), index=False)
        df_f = pd.DataFrame(_results_fq) if _pilot_n else pd.read_csv(FQRS_OUTPUT_CSV)

        # ---- রিপোর্ট ----
        _n_err = int(df_f["Status"].notna().sum()) if "Status" in df_f.columns else 0
        print(f"\n✅ সম্পন্ন। Status-error: {_n_err}/{len(df_f)}")
        try:
            _m = df_meta_fq.copy()
            _m["ecg_id"] = _m["ecg_id"].astype(str).str.replace(r"\D", "", regex=True).astype(int)
            df_f["_label"] = df_f["ECG_ID"].astype(str).str.replace(r"\D", "", regex=True).astype(int).map(
                _m.set_index("ecg_id")["primary_label"])
        except Exception:
            df_f["_label"] = np.nan

        if "fQRS_Territory_Flag" in df_f.columns:
            _v = df_f["fQRS_Territory_Flag"]
            print(f"\n=== fQRS ভূখণ্ড-ফ্ল্যাগ (সরু-QRS রেকর্ডে) ===")
            print(f"মূল্যায়নযোগ্য (সরু-QRS): {_v.notna().mean()*100:.1f}% | "
                  f"flag=1: {_v.mean()*100:.1f}%")
            print(f"per-lead flag গড় (মূল্যায়নযোগ্যতা সহ): "
                  f"{np.nanmean([df_f[f'{L}_fQRS_Flag'].mean() for L in LEAD_NAMES if f'{L}_fQRS_Flag' in df_f.columns])*100:.1f}%")
            if df_f["_label"].notna().any():
                _g = df_f.groupby("_label")["fQRS_Territory_Flag"].agg(["mean", "size"])
                _g["হার_%"] = (_g["mean"] * 100).round(1)
                print(_g[["হার_%", "size"]].to_string())
            print("\n   ক্লিনিক্যাল প্রত্যাশা: MI সর্বোচ্চ (দাগ-fragmentation), NORM নিম্ন (আদর্শ ≤~8–10%)।")
            print("   CD কমও আসতে পারে — প্রশস্ত-QRS রেকর্ড সংজ্ঞামতো মূল্যায়ন-বহির্ভূত (NaN)।")

        if not _pilot_n:
            print(f"\n💾 সেভ: {FQRS_OUTPUT_CSV}")
            print("   পরবর্তী ধাপ: MODEL_ready-তে merge করতে যে সেল দেওয়া হয়েছে, সেটি চালাও।")



# =============================================================================
# STAGE D — MERGE fQRS FEATURES
# =============================================================================
print("\n[4/5] Merging the fQRS add-on into the model-ready table...")
df_model = pd.read_csv(MODEL_READY_CSV)
df_fqrs = pd.read_csv(FQRS_CSV_ONE_STEP)

# Remove extraction-status/helper columns; retain only ECG_ID + fQRS variables.
fqrs_drop = [c for c in ["Status", "_label", "primary_label"] if c in df_fqrs.columns]
df_fqrs = df_fqrs.drop(columns=fqrs_drop)
df_fqrs["ECG_ID"] = (
    df_fqrs["ECG_ID"].astype(str).str.replace(r"\D", "", regex=True).astype(int)
)
df_fqrs = df_fqrs.drop_duplicates("ECG_ID", keep="first")

# Prevent accidental duplicate columns on rerun.
_existing_fqrs = [c for c in df_fqrs.columns if c != "ECG_ID" and c in df_model.columns]
if _existing_fqrs:
    df_model = df_model.drop(columns=_existing_fqrs)

df_final = df_model.merge(df_fqrs, on="ECG_ID", how="left", validate="one_to_one")

# =============================================================================
# STAGE E — ADD patient_id + OFFICIAL strat_fold AND CREATE FINAL FILE
# =============================================================================
print("\n[5/5] Adding patient_id and official PTB-XL strat_fold...")

ptb_meta = pd.read_csv(
    PTBXL_DATABASE_CSV,
    usecols=["ecg_id", "patient_id", "strat_fold"]
)

# Safety: one metadata row per ECG
if ptb_meta["ecg_id"].duplicated().any():
    raise RuntimeError("Duplicate ecg_id found in PTB-XL metadata.")

ptb_meta["ecg_id"] = (
    ptb_meta["ecg_id"]
    .astype(str)
    .str.replace(r"\D", "", regex=True)
    .astype(int)
)

# Remove existing columns if present
for col in ["patient_id", "strat_fold"]:
    if col in df_final.columns:
        df_final = df_final.drop(columns=[col])

# Merge patient_id + strat_fold using ECG_ID
df_final = df_final.merge(
    ptb_meta,
    left_on="ECG_ID",
    right_on="ecg_id",
    how="left",
    validate="one_to_one"
).drop(columns=["ecg_id"])

# Safety checks
if df_final["patient_id"].isna().any():
    missing_ids = df_final.loc[
        df_final["patient_id"].isna(), "ECG_ID"
    ].tolist()[:20]
    raise RuntimeError(
        f"Missing patient_id for ECG IDs: {missing_ids}"
    )

if df_final["strat_fold"].isna().any():
    missing_ids = df_final.loc[
        df_final["strat_fold"].isna(), "ECG_ID"
    ].tolist()[:20]
    raise RuntimeError(
        f"Missing strat_fold for ECG IDs: {missing_ids}"
    )

df_final["patient_id"] = df_final["patient_id"].astype(int)
df_final["strat_fold"] = df_final["strat_fold"].astype(int)

# Final ordering
front = ["ECG_ID", "patient_id", "strat_fold"]
df_final = df_final[
    front + [c for c in df_final.columns if c not in front]
]

df_final = df_final.sort_values("ECG_ID").reset_index(drop=True)

df_final = df_final.sort_values("ECG_ID").reset_index(drop=True)

df_final.to_csv(FINAL_OUTPUT_CSV, index=False)

# =============================================================================
# FINAL AUDIT
# =============================================================================
audit = {
    "pipeline_id": PIPELINE_ID,
    "metadata_records": int(len(_meta_full)),
    "processed_record_ids": int(len(_processed)),
    "successful_master_rows": int(_n_before_hr),
    "hr_consistency_records_removed": int(_n_hr_removed),
    "model_ready_rows": int(len(df_model)),
    "final_rows": int(len(df_final)),
    "final_columns": int(df_final.shape[1]),
    "dropped_columns_present": _dropped_present,
    "missing_fqrs_cells": int(df_final[[c for c in df_fqrs.columns if c != "ECG_ID"]].isna().sum().sum()),
    "output": FINAL_OUTPUT_CSV,
}

# Expected shape check. Raise instead of silently producing a different dataset.
if len(df_final) != EXPECTED_FINAL_ROWS or df_final.shape[1] != EXPECTED_FINAL_COLUMNS:
    audit["shape_warning"] = (
        f"Expected ({EXPECTED_FINAL_ROWS}, {EXPECTED_FINAL_COLUMNS}) but obtained "
        f"{df_final.shape}. Inspect the audit before modeling."
    )
    with open(AUDIT_JSON, "w", encoding="utf-8") as f:
        json.dump(audit, f, indent=2, ensure_ascii=False)
    raise RuntimeError(audit["shape_warning"])

# Optional exact comparison against a differently named previous final file.
if REFERENCE_FINAL_CSV and os.path.exists(REFERENCE_FINAL_CSV):
    ref = pd.read_csv(REFERENCE_FINAL_CSV, low_memory=False)
    audit["reference_shape"] = list(ref.shape)
    audit["same_column_order_as_reference"] = list(ref.columns) == list(df_final.columns)
    audit["same_ecg_id_order_as_reference"] = ref["ECG_ID"].tolist() == df_final["ECG_ID"].tolist()
    common_numeric = [c for c in df_final.columns if c in ref.columns and
                      pd.api.types.is_numeric_dtype(df_final[c]) and
                      pd.api.types.is_numeric_dtype(ref[c])]
    max_diff = 0.0
    for c in common_numeric:
        a = pd.to_numeric(df_final[c], errors="coerce").to_numpy(float)
        b = pd.to_numeric(ref[c], errors="coerce").to_numpy(float)
        if len(a) == len(b):
            d = np.nanmax(np.abs(a-b)) if np.any(np.isfinite(a-b)) else 0.0
            max_diff = max(max_diff, float(d))
    audit["reference_max_absolute_numeric_difference"] = max_diff

with open(AUDIT_JSON, "w", encoding="utf-8") as f:
    json.dump(audit, f, indent=2, ensure_ascii=False)

print("\n" + "="*76)
print("ONE-STEP FINAL FILE COMPLETED")
print("="*76)
print(f"Final shape : {df_final.shape}")
print(f"Final file  : {FINAL_OUTPUT_CSV}")
print(f"Audit file  : {AUDIT_JSON}")
print("Class counts:")
print(df_final["Disease_Name"].value_counts(dropna=False).to_string())
print("Fold counts:")
print(df_final["strat_fold"].value_counts().sort_index().to_string())


## 4. Verification

Compares the column sets of the master file, the fQRS add-on and the final model-ready file. Expected: 0 fQRS columns missing from FINAL, and the drop list matches the paper's QC stage.

In [ ]:
import pandas as pd

MASTER = "/content/drive/MyDrive/PTB-XL Dataset Analysis/ECG_Master_Features_Fixed_File_FINAL.csv"
FQRS   = "/content/drive/MyDrive/PTB-XL Dataset Analysis/ECG_fQRS_Features.csv"
MODEL  = "/content/drive/MyDrive/PTB-XL Dataset Analysis/ECG_MODEL_ready_FINAL_with_StratFold.csv"

master_cols = pd.read_csv(MASTER, nrows=0).columns.tolist()
fqrs_cols   = pd.read_csv(FQRS, nrows=0).columns.tolist()
model_cols  = pd.read_csv(MODEL, nrows=0).columns.tolist()

print("MASTER:", len(master_cols))
print("fQRS:", len(fqrs_cols))
print("FINAL:", len(model_cols))

print("\n=== fQRS columns ===")
for i, c in enumerate(fqrs_cols, 1):
    print(i, c)

print("\n=== FINAL columns ===")
for i, c in enumerate(model_cols, 1):
    print(i, c)

print("\n=== Columns in fQRS but NOT in FINAL ===")
missing_from_final = [c for c in fqrs_cols if c not in model_cols]
print(len(missing_from_final))
for c in missing_from_final:
    print("MISSING:", c)

In [ ]:
print("\n=== MASTER columns NOT in FINAL ===")

master_missing = [c for c in master_cols if c not in model_cols]

print("Count:", len(master_missing))

for c in master_missing:
    print("NOT IN FINAL:", c)